# AANS Podium-to-Print Analysis (2010 vs 2019)

This notebook reproduces the tables and figures used in the manuscript:

**Podium-to-print publication of AANS abstracts fell from 2010 to 2019 despite increased medical student first authorship.**

Outputs written to `AANS_OUT_DIR` (default: `outputs/`):

- Table 1: cohort characteristics (`table1_aans_clean.*`)
- Table 2: logistic regression predictors (`table2_aans_pub_predictors.*`)
- Table 3: publication characteristics among published abstracts (`table3_publication_characteristics_revised.*`)
- Figure 1: Kaplan–Meier cumulative publication probability within 36 months (`figure1_km_*`)
- Figure 2: 2-panel proportions plot (`fig2_clean_main.*`)
- Optional: adjusted OR forest plot (`Figure2_adjusted_OR_forestplot_clean_aligned.*`)

Notes:
- The notebook expects the analysis-ready CSV file (column names used below).
- If you cannot redistribute the CSV, you can still share this notebook and instructions to regenerate the CSV from public sources.


In [27]:
# ----------------------------
# Setup: paths and packages
# ----------------------------
options(stringsAsFactors = FALSE)
options(repos = c(CRAN = "https://cloud.r-project.org"))

install_if_missing <- function(pkgs) {
  for (p in pkgs) {
    if (!requireNamespace(p, quietly = TRUE)) {
      try(install.packages(p, quiet = TRUE), silent = TRUE)
    }
  }
}

# Where to find the analysis-ready CSV (set env var AANS_DATA_PATH to override)
AANS_DATA_PATH <- Sys.getenv("AANS_DATA_PATH", unset = "/home/asegura/repos/aans-abstract-publication-2010-2019/data/AANScleanedv3 JMR 2.25.csv")

# Where outputs are written (set env var AANS_OUT_DIR to override)
AANS_OUT_DIR <- Sys.getenv("AANS_OUT_DIR", unset = "/home/asegura/repos/aans-abstract-publication-2010-2019/outputs")
dir.create(AANS_OUT_DIR, recursive = TRUE, showWarnings = FALSE)

resolve_data_path <- function(path) {
  candidates <- c(
    path,
    "data/AANScleanedv3_JMR_2.25.csv",
    "data/AANScleanedv3 JMR 2.25.csv",
    "AANScleanedv3_JMR_2.25.csv",
    "AANScleanedv3 JMR 2.25.csv"
  )
  candidates <- unique(candidates[!is.na(candidates) & nzchar(candidates)])
  hit <- candidates[file.exists(candidates)][1]
  if (is.na(hit)) {
    stop(
      "Input CSV not found. Set AANS_DATA_PATH or place the CSV at one of:\n- ",
      paste(candidates, collapse = "\n- ")
    )
  }
  normalizePath(hit, winslash = "/", mustWork = TRUE)
}

AANS_DATA_PATH <- resolve_data_path(AANS_DATA_PATH)

install_if_missing(c(
  "dplyr", "ggplot2", "scales", "patchwork", "survival", "readr",
  "officer", "flextable"
))

cat("Using input CSV:\n- ", AANS_DATA_PATH, "\n", sep = "")
cat("Writing outputs to:\n- ", normalizePath(AANS_OUT_DIR, winslash = "/", mustWork = FALSE), "\n", sep = "")


also installing the dependencies ‘ragg’, ‘xml2’


Warning message in install.packages(p, quiet = TRUE):
“installation of package ‘ragg’ had non-zero exit status”
Warning message in install.packages(p, quiet = TRUE):
“installation of package ‘xml2’ had non-zero exit status”
Warning message in install.packages(p, quiet = TRUE):
“installation of package ‘officer’ had non-zero exit status”
also installing the dependencies ‘officer’, ‘ragg’, ‘xml2’


Warning message in install.packages(p, quiet = TRUE):
“installation of package ‘ragg’ had non-zero exit status”
Warning message in install.packages(p, quiet = TRUE):
“installation of package ‘xml2’ had non-zero exit status”
Warning message in install.packages(p, quiet = TRUE):
“installation of package ‘officer’ had non-zero exit status”
Warning message in install.packages(p, quiet = TRUE):
“installation of package ‘flextable’ had non-zero exit status”


Using input CSV:
- /home/asegura/repos/aans-abstract-publication-2010-2019/data/AANScleanedv3 JMR 2.25.csv
Writing outputs to:
- /home/asegura/repos/aans-abstract-publication-2010-2019/outputs


## Table 1
Cohort characteristics of included AANS podium abstracts (2010 vs 2019).

In [28]:
# ============================================================
# Clean Table 1 (no blank cells) with DOCX if possible
# Fallback: HTML (open in Word) + CSV
# Input:  <AANS_DATA_PATH>
# Outputs:
#   - table1_aans_clean.csv
#   - table1_aans_clean.html
#   - table1_aans_clean.docx      (if officer+flextable install)
# ============================================================

DATA_PATH <- AANS_DATA_PATH
OUT_PREFIX <- file.path(AANS_OUT_DIR, "table1_aans_clean")
FONT_NAME <- "Times New Roman"   # change if you want
FONT_SIZE <- 10

options(stringsAsFactors = FALSE)
options(repos = c(CRAN = "https://cloud.r-project.org"))

# ---------- helper: safe install ----------
install_if_missing <- function(pkgs) {
  for (p in pkgs) {
    if (!requireNamespace(p, quietly = TRUE)) {
      try(install.packages(p, quiet = TRUE), silent = TRUE)
    }
  }
}

# ---------- read + validate ----------
stopifnot(file.exists(DATA_PATH))
df0 <- read.csv(DATA_PATH, check.names = FALSE)

required_cols <- c(
  "Year",
  "Subspecialty",
  "Medstudentfirstauthor1Yes0No",
  "Seniorauthorcountry",
  "Numberofauthorsintheabstract",
  "FirstauthorgenderabstractMF",
  "SeniorauthorgenderofabstractMF",
  "Genderconcordantfirstandseniorofabstract1Yes0No",
  "Award",
  "Published",
  "Timetopublicationmonthstimefromabstractapril2019"
)
missing_cols <- setdiff(required_cols, names(df0))
if (length(missing_cols) > 0) {
  stop(paste0("Missing columns:\n- ", paste(missing_cols, collapse = "\n- ")))
}

# ---------- recode helpers ----------
as01 <- function(x) {
  if (is.factor(x)) x <- as.character(x)
  if (is.logical(x)) return(ifelse(is.na(x), NA_integer_, ifelse(x, 1L, 0L)))
  if (is.character(x)) {
    xt <- trimws(x)
    if (xt %in% c("", "NA", "NaN")) return(NA_integer_)
    xn <- suppressWarnings(as.numeric(xt))
    return(ifelse(is.na(xn), NA_integer_, ifelse(xn == 1, 1L, 0L)))
  }
  xn <- suppressWarnings(as.numeric(x))
  ifelse(is.na(xn), NA_integer_, ifelse(xn == 1, 1L, 0L))
}

clean_year <- function(y) {
  yn <- suppressWarnings(as.numeric(y))
  if (all(na.omit(yn) %in% c(0, 1))) {
    ifelse(yn == 0, "2010", ifelse(yn == 1, "2019", NA_character_))
  } else if (all(na.omit(yn) %in% c(2010, 2019))) {
    as.character(yn)
  } else {
    as.character(y)
  }
}

# ---------- formatting helpers ----------
NA_MARK <- "-"  # em dash

fmt_p <- function(p) {
  if (is.na(p)) return(NA_MARK)
  if (p < 0.001) return("<0.001")
  sprintf("%.3f", p)
}

fmt_es <- function(x) {
  if (is.na(x)) return(NA_MARK)
  sprintf("%.3f", x)
}

fmt_n_pct <- function(n, denom) {
  if (is.na(n) || is.na(denom) || denom == 0) return(NA_MARK)
  sprintf("%d (%.1f%%)", n, 100 * n / denom)
}

fmt_median_iqr <- function(x, digits = 1) {
  x <- x[is.finite(x)]
  if (length(x) == 0) return(NA_MARK)
  q <- stats::quantile(x, probs = c(0.25, 0.5, 0.75), na.rm = TRUE, names = FALSE)
  sprintf(paste0("%.", digits, "f (%.", digits, "f-%.", digits, "f)"), q[2], q[1], q[3])
}

# ---------- stats helpers ----------
pval_2x2 <- function(x, g) {
  ok <- !is.na(x) & !is.na(g)
  x <- x[ok]; g <- g[ok]
  tab <- table(x, g)
  if (nrow(tab) != 2 || ncol(tab) != 2) return(NA_real_)
  exp <- suppressWarnings(stats::chisq.test(tab, correct = FALSE)$expected)
  if (any(exp < 5)) return(stats::fisher.test(tab)$p.value)
  stats::chisq.test(tab, correct = FALSE)$p.value
}

pval_cat_global <- function(x, g) {
  ok <- !is.na(x) & !is.na(g)
  x <- x[ok]; g <- g[ok]
  tab <- table(x, g)
  if (length(tab) == 0) return(NA_real_)
  exp <- suppressWarnings(stats::chisq.test(tab, correct = FALSE)$expected)
  if (any(exp < 5)) return(stats::chisq.test(tab, simulate.p.value = TRUE, B = 20000)$p.value)
  stats::chisq.test(tab, correct = FALSE)$p.value
}

cramers_v <- function(x, g) {
  ok <- !is.na(x) & !is.na(g)
  x <- x[ok]; g <- g[ok]
  tab <- table(x, g)
  if (sum(tab) == 0) return(NA_real_)
  chi <- suppressWarnings(stats::chisq.test(tab, correct = FALSE))
  r <- nrow(tab); c <- ncol(tab)
  k <- min(r - 1, c - 1)
  if (k <= 0) return(NA_real_)
  as.numeric(sqrt(chi$statistic / (sum(tab) * k)))
}

asd_binary <- function(x0, x1) {
  x0 <- x0[!is.na(x0)]; x1 <- x1[!is.na(x1)]
  if (length(x0) == 0 || length(x1) == 0) return(NA_real_)
  p0 <- mean(x0 == 1); p1 <- mean(x1 == 1)
  p  <- (sum(x0 == 1) + sum(x1 == 1)) / (length(x0) + length(x1))
  denom <- sqrt(p * (1 - p))
  if (!is.finite(denom) || denom == 0) return(0)
  abs((p1 - p0) / denom)
}

asd_cont <- function(x0, x1) {
  x0 <- x0[is.finite(x0)]; x1 <- x1[is.finite(x1)]
  if (length(x0) == 0 || length(x1) == 0) return(NA_real_)
  m0 <- mean(x0); m1 <- mean(x1)
  s0 <- stats::sd(x0); s1 <- stats::sd(x1)
  sp <- sqrt(((length(x0) - 1) * s0^2 + (length(x1) - 1) * s1^2) / (length(x0) + length(x1) - 2))
  if (!is.finite(sp) || sp == 0) return(0)
  abs((m1 - m0) / sp)
}

pval_wilcox <- function(x, g) {
  ok <- is.finite(x) & !is.na(g)
  x <- x[ok]; g <- g[ok]
  if (length(unique(g)) != 2) return(NA_real_)
  stats::wilcox.test(x ~ g)$p.value
}

# ---------- recode columns ----------
df <- df0
df$Year_lab <- clean_year(df$Year)

# Subspecialty mapping (edit if your codebook differs)
sub_map <- c(
  "1" = "Functional/Peripheral Nerve",
  "2" = "Pediatrics",
  "3" = "Spine",
  "4" = "Vascular/Stroke",
  "5" = "General/Other",
  "6" = "Trauma/Critical Care",
  "7" = "Tumor"
)
df$Subspecialty_lab <- trimws(as.character(df$Subspecialty))
sub_num <- suppressWarnings(as.numeric(df$Subspecialty_lab))
if (all(na.omit(sub_num) %in% suppressWarnings(as.numeric(names(sub_map))))) {
  df$Subspecialty_lab <- unname(sub_map[as.character(sub_num)])
}

df$med_student_first    <- as01(df$Medstudentfirstauthor1Yes0No)
df$senior_international <- as01(df$Seniorauthorcountry)
df$n_authors            <- suppressWarnings(as.numeric(df$Numberofauthorsintheabstract))

# gender vars assumed 0=Male, 1=Female
df$first_female      <- as01(df$FirstauthorgenderabstractMF)
df$senior_female     <- as01(df$SeniorauthorgenderofabstractMF)
df$gender_concordant <- as01(df$Genderconcordantfirstandseniorofabstract1Yes0No)

df$award     <- as01(df$Award)
df$published <- as01(df$Published)

df$time_to_pub_mo <- suppressWarnings(as.numeric(df$Timetopublicationmonthstimefromabstractapril2019))
df$time_to_pub_mo[df$published != 1] <- NA_real_

df <- df[df$Year_lab %in% c("2010", "2019"), , drop = FALSE]
g2010 <- df[df$Year_lab == "2010", , drop = FALSE]
g2019 <- df[df$Year_lab == "2019", , drop = FALSE]

n_all  <- nrow(df)
n_2010 <- nrow(g2010)
n_2019 <- nrow(g2019)

col_overall <- sprintf("Overall (n=%d)", n_all)
col_2010    <- sprintf("2010 (n=%d)", n_2010)
col_2019    <- sprintf("2019 (n=%d)", n_2019)

# denominators for subspecialty % (handle missing subspecialty)
den_sub_all  <- sum(!is.na(df$Subspecialty_lab))
den_sub_2010 <- sum(!is.na(g2010$Subspecialty_lab))
den_sub_2019 <- sum(!is.na(g2019$Subspecialty_lab))

# ---------- build table rows (NO blanks) ----------
rows <- list()
add_row <- function(ch, o, y0, y1, es, pv) {
  rows[[length(rows) + 1]] <<- data.frame(
    Characteristic = ch,
    o = o, y0 = y0, y1 = y1, es = es, pv = pv,
    check.names = FALSE
  )
}

# Row: N abstracts
add_row("Number of abstracts", sprintf("%d", n_all), sprintf("%d", n_2010), sprintf("%d", n_2019), NA_MARK, NA_MARK)

# Subspecialty global row (distribution test)
v_sub <- cramers_v(df$Subspecialty_lab, df$Year_lab)
p_sub <- pval_cat_global(df$Subspecialty_lab, df$Year_lab)
add_row("Subspecialty, n (%)", NA_MARK, NA_MARK, NA_MARK, paste0("V=", fmt_es(v_sub)), fmt_p(p_sub))

# Subspecialty levels: fixed order (matches your screenshot), with fallbacks
sub_order <- c(
  "Functional/Peripheral Nerve",
  "General/Other",
  "Pediatrics",
  "Spine",
  "Trauma/Critical Care",
  "Tumor",
  "Vascular/Stroke"
)
sub_levels <- sub_order[sub_order %in% unique(df$Subspecialty_lab)]
# include any unexpected labels (if present)
extra_levels <- setdiff(unique(df$Subspecialty_lab), sub_levels)
sub_levels <- c(sub_levels, sort(extra_levels))

for (lvl in sub_levels) {
  add_row(
    paste0("  ", lvl),
    fmt_n_pct(sum(df$Subspecialty_lab == lvl,  na.rm = TRUE), den_sub_all),
    fmt_n_pct(sum(g2010$Subspecialty_lab == lvl, na.rm = TRUE), den_sub_2010),
    fmt_n_pct(sum(g2019$Subspecialty_lab == lvl, na.rm = TRUE), den_sub_2019),
    NA_MARK,
    NA_MARK
  )
}

add_binary <- function(var_all, var_2010, var_2019, label) {
  den_all  <- sum(!is.na(var_all))
  den_2010 <- sum(!is.na(var_2010))
  den_2019 <- sum(!is.na(var_2019))
  n_yes_all  <- sum(var_all  == 1, na.rm = TRUE)
  n_yes_2010 <- sum(var_2010 == 1, na.rm = TRUE)
  n_yes_2019 <- sum(var_2019 == 1, na.rm = TRUE)
  add_row(
    paste0(label, ", n (%)"),
    fmt_n_pct(n_yes_all,  den_all),
    fmt_n_pct(n_yes_2010, den_2010),
    fmt_n_pct(n_yes_2019, den_2019),
    fmt_es(asd_binary(var_2010, var_2019)),
    fmt_p(pval_2x2(var_all, df$Year_lab))
  )
}

add_binary(df$med_student_first,    g2010$med_student_first,    g2019$med_student_first,    "Medical student first author")
add_binary(df$senior_international, g2010$senior_international, g2019$senior_international, "International senior author")

add_row(
  "Number of authors in abstract, median (IQR)",
  fmt_median_iqr(df$n_authors,  digits = 1),
  fmt_median_iqr(g2010$n_authors, digits = 1),
  fmt_median_iqr(g2019$n_authors, digits = 1),
  fmt_es(asd_cont(g2010$n_authors, g2019$n_authors)),
  fmt_p(pval_wilcox(df$n_authors, df$Year_lab))
)

add_binary(df$first_female,      g2010$first_female,      g2019$first_female,      "Female first author (abstract; inferred/perceived)")
add_binary(df$senior_female,     g2010$senior_female,     g2019$senior_female,     "Female senior author (abstract; inferred/perceived)")
add_binary(df$gender_concordant, g2010$gender_concordant, g2019$gender_concordant, "Gender concordance (first vs senior; abstract)")
add_binary(df$award,             g2010$award,             g2019$award,             "Award-winning abstract")
add_binary(df$published,         g2010$published,         g2019$published,         "Published (podium-to-print)")

add_row(
  "Time to publication (months), median (IQR) [published only]",
  fmt_median_iqr(df$time_to_pub_mo, digits = 1),
  fmt_median_iqr(g2010$time_to_pub_mo, digits = 1),
  fmt_median_iqr(g2019$time_to_pub_mo, digits = 1),
  fmt_es(asd_cont(g2010$time_to_pub_mo, g2019$time_to_pub_mo)),
  fmt_p(pval_wilcox(df$time_to_pub_mo, df$Year_lab))
)

tab <- do.call(rbind, rows)

# replace any accidental NA/"" with em dash
for (j in seq_len(ncol(tab))) {
  tab[[j]][is.na(tab[[j]]) | tab[[j]] == ""] <- NA_MARK
}

# Final column names
names(tab) <- c("Characteristic", col_overall, col_2010, col_2019, "Effect size", "P-value")

# ---------- export CSV ----------
write.csv(tab, paste0(OUT_PREFIX, ".csv"), row.names = FALSE)

# ---------- export HTML (Word can open this) ----------
html_escape <- function(x) {
  x <- gsub("&", "&amp;", x, fixed = TRUE)
  x <- gsub("<", "&lt;", x, fixed = TRUE)
  x <- gsub(">", "&gt;", x, fixed = TRUE)
  x
}

is_indent <- grepl("^  ", tab$Characteristic)
is_block  <- tab$Characteristic == "Subspecialty, n (%)"

tab_html <- tab
tab_html$Characteristic <- sub("^  ", "", tab_html$Characteristic)

css <- sprintf("
body { font-family: '%s'; font-size: %dpt; }
table { border-collapse: collapse; width: 100%%; }
caption { caption-side: top; text-align: left; font-weight: bold; margin-bottom: 6pt; }
th, td { padding: 4pt 6pt; vertical-align: top; }
th { text-align: left; border-bottom: 0.5pt solid #000; }
table { border-top: 1pt solid #000; border-bottom: 1pt solid #000; }
td.num { text-align: center; }
td.es, td.pv { text-align: right; white-space: nowrap; }
tr.block td { font-weight: bold; padding-top: 6pt; }
tr.sep td { border-bottom: 0.5pt solid #000; }
td.indent { padding-left: 14pt; }
.notes { margin-top: 8pt; font-size: 9pt; }
", FONT_NAME, FONT_SIZE)

# build rows
make_row <- function(i) {
  cls <- ""
  if (is_block[i]) cls <- " class='block'"
  # add separator after last subspecialty level
  sep_cls <- ""
  # identify last indented row index
  last_sub <- if (any(is_indent)) max(which(is_indent)) else -1
  if (i == last_sub) sep_cls <- " class='sep'"

  tr_open <- if (nzchar(cls)) paste0("<tr", cls, ">") else if (nzchar(sep_cls)) paste0("<tr", sep_cls, ">") else "<tr>"
  if (nzchar(cls) && nzchar(sep_cls)) tr_open <- "<tr class='block sep'>"

  ch_td_class <- if (is_indent[i]) "indent" else ""
  sprintf(
    "%s<td class='%s'>%s</td><td class='num'>%s</td><td class='num'>%s</td><td class='num'>%s</td><td class='es'>%s</td><td class='pv'>%s</td></tr>",
    tr_open,
    ch_td_class,
    html_escape(tab_html$Characteristic[i]),
    html_escape(tab_html[[col_overall]][i]),
    html_escape(tab_html[[col_2010]][i]),
    html_escape(tab_html[[col_2019]][i]),
    html_escape(tab_html[["Effect size"]][i]),
    html_escape(tab_html[["P-value"]][i])
  )
}

html <- c(
  "<html><head><meta charset='utf-8'/>",
  "<style>", css, "</style></head><body>",
  "<table>",
  "<caption>Table 1. Characteristics of AANS podium presentation abstracts (2010 vs 2019)</caption>",
  sprintf(
    "<tr><th>Characteristic</th><th>%s</th><th>%s</th><th>%s</th><th>Effect size</th><th>P-value</th></tr>",
    html_escape(col_overall), html_escape(col_2010), html_escape(col_2019)
  ),
  vapply(seq_len(nrow(tab)), make_row, character(1)),
  "</table>",
  "<div class='notes'>Notes: — indicates not applicable. Effect size uses ASD for binary/continuous variables and Cramer's V (shown as V=...) for subspecialty overall. Gender variables reflect inferred/perceived gender (proxy), not self-identified gender.</div>",
  "</body></html>"
)

writeLines(html, paste0(OUT_PREFIX, ".html"))

# ---------- DOCX (optional) ----------
install_if_missing(c("officer", "flextable"))

have_docx <- requireNamespace("officer", quietly = TRUE) && requireNamespace("flextable", quietly = TRUE)

if (have_docx) {
  library(officer)
  library(flextable)

  ft <- flextable(tab)
  ft <- font(ft, fontname = FONT_NAME, part = "all")
  ft <- fontsize(ft, size = FONT_SIZE, part = "all")
  ft <- bold(ft, part = "header")

  # alignment
  ft <- align(ft, j = 1, align = "left", part = "all")
  ft <- align(ft, j = 2:4, align = "center", part = "all")
  ft <- align(ft, j = 5:6, align = "right", part = "all")

  # indent subspecialty rows
  sub_i <- which(grepl("^  ", tab$Characteristic))
  if (length(sub_i) > 0) ft <- padding(ft, i = sub_i, j = 1, padding.left = 12, part = "body")

  # bold block header
  block_i <- which(tab$Characteristic == "Subspecialty, n (%)")
  if (length(block_i) == 1) ft <- bold(ft, i = block_i, part = "body")

  # borders: remove verticals, minimal horizontals
  ft <- border_remove(ft)
  b_top <- fp_border(color = "black", width = 1)
  b_mid <- fp_border(color = "black", width = 0.5)
  ft <- hline_top(ft, part = "all", border = b_top)
  ft <- hline_bottom(ft, part = "header", border = b_mid)
  ft <- hline_bottom(ft, part = "body", border = b_top)

  # separator after subspecialty block
  if (length(sub_i) > 0) ft <- hline(ft, i = max(sub_i), part = "body", border = b_mid)

  ft <- autofit(ft)

  ft <- add_footer_lines(
    ft,
    values = c(
      "Notes: — indicates not applicable.",
      "Effect size: ASD for binary/continuous variables; Cramer's V for subspecialty overall.",
      "Gender variables reflect inferred/perceived gender (proxy), not self-identified gender."
    )
  )

  save_as_docx(`Table 1` = ft, path = paste0(OUT_PREFIX, ".docx"))
  cat("Wrote DOCX:", paste0(OUT_PREFIX, ".docx"), "\n")
} else {
  cat("Could not install/load officer+flextable; wrote HTML instead.\n",
      "Open ", OUT_PREFIX, ".html in Word and save as .docx.\n", sep = "")
}

cat("Wrote:\n- ", OUT_PREFIX, ".csv\n- ", OUT_PREFIX, ".html\n", sep = "")

also installing the dependencies ‘ragg’, ‘xml2’


Warning message in install.packages(p, quiet = TRUE):
“installation of package ‘ragg’ had non-zero exit status”
Warning message in install.packages(p, quiet = TRUE):
“installation of package ‘xml2’ had non-zero exit status”
Warning message in install.packages(p, quiet = TRUE):
“installation of package ‘officer’ had non-zero exit status”
also installing the dependencies ‘officer’, ‘ragg’, ‘xml2’


Warning message in install.packages(p, quiet = TRUE):
“installation of package ‘ragg’ had non-zero exit status”
Warning message in install.packages(p, quiet = TRUE):
“installation of package ‘xml2’ had non-zero exit status”
Warning message in install.packages(p, quiet = TRUE):
“installation of package ‘officer’ had non-zero exit status”
Warning message in install.packages(p, quiet = TRUE):
“installation of package ‘flextable’ had non-zero exit status”


Could not install/load officer+flextable; wrote HTML instead.
Open /home/asegura/repos/aans-abstract-publication-2010-2019/outputs/table1_aans_clean.html in Word and save as .docx.
Wrote:
- /home/asegura/repos/aans-abstract-publication-2010-2019/outputs/table1_aans_clean.csv
- /home/asegura/repos/aans-abstract-publication-2010-2019/outputs/table1_aans_clean.html


## Table 2
Predictors of podium-to-print publication (univariable and multivariable logistic regression).

In [29]:
# ============================================================
# Table 2 (Manuscript): Predictors of podium-to-print publication
# - Univariable + multivariable logistic regression
# - Global LR test row for Subspecialty
# - Professional HTML export (open in Word -> Save As .docx)
#
# INPUT:
#   <AANS_DATA_PATH>
#
# OUTPUT:
#   table2_aans_pub_predictors.csv
#   table2_aans_pub_predictors.html
# ============================================================

DATA_PATH <- AANS_DATA_PATH
OUT_PREFIX <- file.path(AANS_OUT_DIR, "table2_aans_pub_predictors")

stopifnot(file.exists(DATA_PATH))

options(stringsAsFactors = FALSE)

df0 <- read.csv(DATA_PATH, check.names = FALSE)

required_cols <- c(
  "Year",
  "Subspecialty",
  "Medstudentfirstauthor1Yes0No",
  "Seniorauthorcountry",
  "Numberofauthorsintheabstract",
  "FirstauthorgenderabstractMF",
  "SeniorauthorgenderofabstractMF",
  "Award",
  "Published"
)
missing_cols <- setdiff(required_cols, names(df0))
if (length(missing_cols) > 0) {
  stop(paste0("Missing columns:\n- ", paste(missing_cols, collapse = "\n- ")))
}

# -------------------------
# Helpers
# -------------------------
as01 <- function(x) {
  if (is.factor(x)) x <- as.character(x)
  if (is.logical(x)) return(ifelse(is.na(x), NA_integer_, ifelse(x, 1L, 0L)))
  if (is.character(x)) {
    xt <- trimws(x)
    if (xt %in% c("", "NA", "NaN")) return(NA_integer_)
    xn <- suppressWarnings(as.numeric(xt))
    return(ifelse(is.na(xn), NA_integer_, ifelse(xn == 1, 1L, 0L)))
  }
  xn <- suppressWarnings(as.numeric(x))
  ifelse(is.na(xn), NA_integer_, ifelse(xn == 1, 1L, 0L))
}

clean_year <- function(y) {
  yn <- suppressWarnings(as.numeric(y))
  if (all(na.omit(yn) %in% c(0, 1))) {
    ifelse(yn == 0, "2010", ifelse(yn == 1, "2019", NA_character_))
  } else if (all(na.omit(yn) %in% c(2010, 2019))) {
    as.character(yn)
  } else {
    as.character(y)
  }
}

fmt_p3 <- function(p) {
  if (is.na(p)) return("NA")
  if (p < 0.001) return("<0.001")
  sprintf("%.3f", p)
}

fmt_or_ci <- function(or, lcl, ucl, digits = 2) {
  if (!is.finite(or) || !is.finite(lcl) || !is.finite(ucl)) return("NA")
  sprintf(paste0("%.", digits, "f (%.", digits, "f-%.", digits, "f)"), or, lcl, ucl)
}

safe_glm <- function(formula, data) {
  tryCatch(glm(formula, family = binomial(), data = data),
           error = function(e) e)
}

get_term_stats <- function(fit, term) {
  if (inherits(fit, "error")) return(list(orci = "NA", p = "NA"))
  if (!inherits(fit, "glm"))  return(list(orci = "NA", p = "NA"))

  co <- summary(fit)$coefficients
  if (is.null(co) || !(term %in% rownames(co))) return(list(orci = "NA", p = "NA"))

  beta <- co[term, "Estimate"]
  se   <- co[term, "Std. Error"]
  p    <- co[term, "Pr(>|z|)"]

  if (!is.finite(beta) || !is.finite(se) || se <= 0) {
    return(list(orci = "NA", p = fmt_p3(p)))
  }

  or  <- exp(beta)
  lcl <- exp(beta - 1.96 * se)
  ucl <- exp(beta + 1.96 * se)
  list(orci = fmt_or_ci(or, lcl, ucl, digits = 2), p = fmt_p3(p))
}

# Likelihood-ratio test p-value for nested models
lr_pvalue <- function(fit_reduced, fit_full) {
  if (inherits(fit_reduced, "error") || inherits(fit_full, "error")) return(NA_real_)
  if (!inherits(fit_reduced, "glm") || !inherits(fit_full, "glm")) return(NA_real_)
  a <- tryCatch(anova(fit_reduced, fit_full, test = "Chisq"), error = function(e) NULL)
  if (is.null(a) || nrow(a) < 2) return(NA_real_)
  as.numeric(a$`Pr(>Chi)`[nrow(a)])
}

# AUC from ranks (no packages)
auc_rank <- function(y, p) {
  ok <- is.finite(p) & !is.na(y)
  y <- y[ok]; p <- p[ok]
  if (length(unique(y)) != 2) return(NA_real_)
  n1 <- sum(y == 1); n0 <- sum(y == 0)
  if (n1 == 0 || n0 == 0) return(NA_real_)
  r <- rank(p, ties.method = "average")
  (sum(r[y == 1]) - n1 * (n1 + 1) / 2) / (n1 * n0)
}

# -------------------------
# Recode for modeling (stable coefficient names)
# -------------------------
df <- df0
df$year <- factor(clean_year(df$Year), levels = c("2010", "2019"))

sub_map <- c(
  "1" = "Functional/Peripheral Nerve",
  "2" = "Pediatrics",
  "3" = "Spine",
  "4" = "Vascular/Stroke",
  "5" = "General/Other",
  "6" = "Trauma/Critical Care",
  "7" = "Tumor"
)
sub_raw <- trimws(as.character(df$Subspecialty))
sub_num <- suppressWarnings(as.numeric(sub_raw))
sub_lab <- sub_raw
if (all(na.omit(sub_num) %in% suppressWarnings(as.numeric(names(sub_map))))) {
  sub_lab <- unname(sub_map[as.character(sub_num)])
}

sub_code_map <- c(
  "General/Other" = "general_other",
  "Functional/Peripheral Nerve" = "functional_peripheral_nerve",
  "Pediatrics" = "pediatrics",
  "Spine" = "spine",
  "Trauma/Critical Care" = "trauma_critical_care",
  "Tumor" = "tumor",
  "Vascular/Stroke" = "vascular_stroke"
)

df$subspec <- unname(sub_code_map[sub_lab])
df$subspec <- factor(
  df$subspec,
  levels = c("general_other",
             "functional_peripheral_nerve",
             "pediatrics",
             "spine",
             "trauma_critical_care",
             "tumor",
             "vascular_stroke")
)

df$med_student <- factor(ifelse(as01(df$Medstudentfirstauthor1Yes0No) == 1, "yes", "no"),
                         levels = c("no", "yes"))

df$senior_country <- factor(ifelse(as01(df$Seniorauthorcountry) == 1, "international", "usa"),
                            levels = c("usa", "international"))

df$n_authors <- suppressWarnings(as.numeric(df$Numberofauthorsintheabstract))

# Gender vars assumed: 0=Male, 1=Female
df$first_gender <- factor(ifelse(as01(df$FirstauthorgenderabstractMF) == 1, "female", "male"),
                          levels = c("male", "female"))
df$senior_gender <- factor(ifelse(as01(df$SeniorauthorgenderofabstractMF) == 1, "female", "male"),
                           levels = c("male", "female"))

df$award <- factor(ifelse(as01(df$Award) == 1, "yes", "no"),
                   levels = c("no", "yes"))

df$published <- as01(df$Published)

# Keep only 2010/2019 + non-missing outcome
df <- df[!is.na(df$year) & df$year %in% c("2010","2019") & !is.na(df$published), , drop = FALSE]

# Complete-case dataset (constant N across unadjusted/adjusted comparisons)
vars_needed <- c("published","year","subspec","med_student","senior_country","n_authors","first_gender","senior_gender","award")
mdata <- df[, vars_needed, drop = FALSE]
mdata <- mdata[complete.cases(mdata), , drop = FALSE]

N_model <- nrow(mdata)
events  <- sum(mdata$published == 1)

# -------------------------
# Fit models
# -------------------------
fit_null <- safe_glm(published ~ 1, data = mdata)

# univariable
fit_uni_year          <- safe_glm(published ~ year, data = mdata)
fit_uni_subspec       <- safe_glm(published ~ subspec, data = mdata)
fit_uni_med_student   <- safe_glm(published ~ med_student, data = mdata)
fit_uni_senior_ctry   <- safe_glm(published ~ senior_country, data = mdata)
fit_uni_n_authors     <- safe_glm(published ~ n_authors, data = mdata)
fit_uni_first_gender  <- safe_glm(published ~ first_gender, data = mdata)
fit_uni_senior_gender <- safe_glm(published ~ senior_gender, data = mdata)
fit_uni_award         <- safe_glm(published ~ award, data = mdata)

# adjusted
fit_adj <- safe_glm(
  published ~ year + subspec + med_student + senior_country + n_authors + first_gender + senior_gender + award,
  data = mdata
)

# Global LR test p-values for subspecialty
p_subspec_uni <- lr_pvalue(fit_null, fit_uni_subspec)

fit_adj_reduced_no_subspec <- safe_glm(
  published ~ year + med_student + senior_country + n_authors + first_gender + senior_gender + award,
  data = mdata
)
p_subspec_adj <- lr_pvalue(fit_adj_reduced_no_subspec, fit_adj)

# Diagnostics (optional, included in notes)
if (inherits(fit_adj, "glm")) {
  p_hat <- predict(fit_adj, type = "response")
  auc <- auc_rank(mdata$published, p_hat)
  mcfadden_r2 <- if (inherits(fit_null, "glm")) {
    1 - (as.numeric(logLik(fit_adj)) / as.numeric(logLik(fit_null)))
  } else NA_real_
} else {
  auc <- NA_real_
  mcfadden_r2 <- NA_real_
}

# -------------------------
# Table 2 specification
# -------------------------
label_map <- c(
  "year2019" = "Year: 2019 vs 2010",

  "SUBSPEC_GLOBAL" = "Subspecialty (global LR test; reference: General/Other)",
  "subspecfunctional_peripheral_nerve" = "  Functional/Peripheral Nerve vs General/Other",
  "subspecpediatrics" = "  Pediatrics vs General/Other",
  "subspecspine" = "  Spine vs General/Other",
  "subspectrauma_critical_care" = "  Trauma/Critical Care vs General/Other",
  "subspectumor" = "  Tumor vs General/Other",
  "subspecvascular_stroke" = "  Vascular/Stroke vs General/Other",

  "med_studentyes" = "Medical student first author: Yes vs No",
  "senior_countryinternational" = "International senior author: International vs USA",
  "n_authors" = "Number of authors in abstract (per +1 author)",
  "first_genderfemale" = "Female first author (abstract): Female vs Male",
  "senior_genderfemale" = "Female senior author (abstract): Female vs Male",
  "awardyes" = "Award-winning abstract: Yes vs No"
)

# Only include subspecialty levels that actually appear in the complete-case data
present_sub_levels <- levels(droplevels(mdata$subspec))
sub_level_terms <- paste0("subspec", present_sub_levels[present_sub_levels != "general_other"])
sub_level_terms <- intersect(
  sub_level_terms,
  c("subspecfunctional_peripheral_nerve","subspecpediatrics","subspecspine",
    "subspectrauma_critical_care","subspectumor","subspecvascular_stroke")
)

row_order <- c(
  "year2019",
  "SUBSPEC_GLOBAL",
  sub_level_terms,
  "med_studentyes",
  "senior_countryinternational",
  "n_authors",
  "first_genderfemale",
  "senior_genderfemale",
  "awardyes"
)

# initialize output table (preserve column names)
table2 <- data.frame(
  Predictor = character(),
  `Unadjusted OR (95% CI)` = character(),
  `P-value` = character(),
  `Adjusted OR (95% CI)` = character(),
  `P-value (adj)` = character(),
  stringsAsFactors = FALSE,
  check.names = FALSE
)

add_row <- function(pred, u_orci, u_p, a_orci, a_p) {
  table2 <<- rbind(
    table2,
    data.frame(
      Predictor = pred,
      `Unadjusted OR (95% CI)` = u_orci,
      `P-value` = u_p,
      `Adjusted OR (95% CI)` = a_orci,
      `P-value (adj)` = a_p,
      stringsAsFactors = FALSE,
      check.names = FALSE
    )
  )
}

uni_model_for_term <- function(term) {
  if (grepl("^year", term)) return(fit_uni_year)
  if (grepl("^subspec", term)) return(fit_uni_subspec)
  if (grepl("^med_student", term)) return(fit_uni_med_student)
  if (grepl("^senior_country", term)) return(fit_uni_senior_ctry)
  if (term == "n_authors") return(fit_uni_n_authors)
  if (grepl("^first_gender", term)) return(fit_uni_first_gender)
  if (grepl("^senior_gender", term)) return(fit_uni_senior_gender)
  if (grepl("^award", term)) return(fit_uni_award)
  fit_uni_year
}

for (term in row_order) {
  if (term == "SUBSPEC_GLOBAL") {
    add_row(
      label_map[[term]],
      "-", fmt_p3(p_subspec_uni),
      "-", fmt_p3(p_subspec_adj)
    )
    next
  }

  uni_fit <- uni_model_for_term(term)
  uni_vals <- get_term_stats(uni_fit, term)
  adj_vals <- get_term_stats(fit_adj, term)

  add_row(
    label_map[[term]],
    uni_vals$orci, uni_vals$p,
    adj_vals$orci, adj_vals$p
  )
}

# Ensure no blanks
for (j in seq_len(ncol(table2))) {
  table2[[j]][is.na(table2[[j]]) | table2[[j]] == ""] <- "NA"
}

# -------------------------
# Export CSV
# -------------------------
write.csv(table2, paste0(OUT_PREFIX, ".csv"), row.names = FALSE)

# -------------------------
# Export HTML (Word-openable, journal-like)
# -------------------------
html_escape <- function(x) {
  x <- as.character(x)
  x <- gsub("&", "&amp;", x, fixed = TRUE)
  x <- gsub("<", "&lt;", x, fixed = TRUE)
  x <- gsub(">", "&gt;", x, fixed = TRUE)
  x
}

is_global <- table2$Predictor == label_map[["SUBSPEC_GLOBAL"]]
is_indent <- grepl("^\\s\\s", table2$Predictor)

css <- "
body { font-family: 'Times New Roman'; font-size: 10pt; }
table { border-collapse: collapse; width: 100%; }
caption { caption-side: top; text-align: left; font-weight: bold; margin-bottom: 6pt; }
th, td { padding: 4pt 6pt; vertical-align: top; }
th { text-align: left; border-bottom: 0.5pt solid #000; }
table { border-top: 1pt solid #000; border-bottom: 1pt solid #000; }
td.c { text-align: center; white-space: nowrap; }
td.r { text-align: right; white-space: nowrap; }
tr.hdr td { font-weight: bold; padding-top: 6pt; }
td.indent { padding-left: 14pt; }
.notes { margin-top: 8pt; font-size: 9pt; }
"

make_row <- function(i) {
  tr_class <- if (isTRUE(is_global[i])) " class='hdr'" else ""
  pred_class <- if (isTRUE(is_indent[i])) "indent" else ""

  pred <- html_escape(sub("^\\s\\s", "", table2[i, 1, drop = TRUE]))
  u_or <- html_escape(table2[i, 2, drop = TRUE])
  u_p  <- html_escape(table2[i, 3, drop = TRUE])
  a_or <- html_escape(table2[i, 4, drop = TRUE])
  a_p  <- html_escape(table2[i, 5, drop = TRUE])

  sprintf(
    "<tr%s><td class='%s'>%s</td><td class='c'>%s</td><td class='r'>%s</td><td class='c'>%s</td><td class='r'>%s</td></tr>",
    tr_class, pred_class, pred, u_or, u_p, a_or, a_p
  )
}

rows_html <- vapply(seq_len(nrow(table2)), make_row, character(1), USE.NAMES = FALSE)

caption <- sprintf(
  "Table 2. Predictors of podium-to-print publication (logistic regression; complete-case N=%d, published n=%d).",
  N_model, events
)

notes <- sprintf(
  paste0(
    "Notes: ORs with 95%% CIs shown. Univariable p-values are Wald tests from univariable logistic regression. ",
    "Adjusted p-values are Wald tests from the multivariable model. Subspecialty (global) p-values are likelihood-ratio tests (reduced vs full). ",
    "P-values are nominal. Model AUC=%.3f; McFadden R2=%.3f. ",
    "Gender variables reflect inferred/perceived gender (proxy), not self-identified gender."
  ),
  auc, mcfadden_r2
)

html <- c(
  "<html><head><meta charset='utf-8'/>",
  "<style>", css, "</style></head><body>",
  "<table>",
  "<caption>", html_escape(caption), "</caption>",
  "<tr>",
  "<th>Predictor</th>",
  "<th>Unadjusted OR (95% CI)</th>",
  "<th>P-value</th>",
  "<th>Adjusted OR (95% CI)</th>",
  "<th>P-value</th>",
  "</tr>",
  rows_html,
  "</table>",
  "<div class='notes'>", html_escape(notes), "</div>",
  "</body></html>"
)

writeLines(html, paste0(OUT_PREFIX, ".html"))

cat("Wrote:\n",
    "- ", OUT_PREFIX, ".csv\n",
    "- ", OUT_PREFIX, ".html (open in Word -> Save As .docx)\n", sep = "")

Wrote:
- /home/asegura/repos/aans-abstract-publication-2010-2019/outputs/table2_aans_pub_predictors.csv
- /home/asegura/repos/aans-abstract-publication-2010-2019/outputs/table2_aans_pub_predictors.html (open in Word -> Save As .docx)


## Table 3
Characteristics of resulting publications among published abstracts only (2010 vs 2019), with raw and Holm-adjusted p-values.

In [30]:
# ============================================================
# Revised Table 3 (published manuscripts only; 2010 vs 2019)
# - Descriptive stats + BOTH raw and Holm-adjusted p-values
# - Holm adjustment applied across the 7 primary comparisons in the table:
#     1) Time to publication
#     2) Journal impact factor
#     3) Number of authors (final publication)
#     4) Author list expansion (pub - abstract)
#     5) Author-count change category (3-level)
#     6) Female first author (publication)
#     7) Female senior author (publication)
# - Removes redundancy: DROPS "Impact factor >= 5" row
#
# INPUT:
#   <AANS_DATA_PATH>
#
# OUTPUT:
#   table3_publication_characteristics_revised.csv
#   table3_publication_characteristics_revised.html   (open in Word -> Save As .docx)
# ============================================================

DATA_PATH <- AANS_DATA_PATH
OUT_PREFIX <- file.path(AANS_OUT_DIR, "table3_publication_characteristics_revised")

stopifnot(file.exists(DATA_PATH))

options(stringsAsFactors = FALSE)
set.seed(1)  # for simulated chi-square p-values (reproducible)

df0 <- read.csv(DATA_PATH, check.names = FALSE)

required_cols <- c(
  "Year","Published",
  "Numberofauthorsintheabstract","Numberofauthorsinthefinalpublication",
  "Impactfactor",
  "Firstauthorgenderofpublication","Seniorauthorgenderofpublication",
  "Timetopublicationmonthstimefromabstractapril2019"
)
missing_cols <- setdiff(required_cols, names(df0))
if (length(missing_cols) > 0) {
  stop(paste0("Missing columns:\n- ", paste(missing_cols, collapse = "\n- ")))
}

# -------------------------
# Helpers
# -------------------------
as01 <- function(x) {
  if (is.factor(x)) x <- as.character(x)
  if (is.logical(x)) return(ifelse(is.na(x), NA_integer_, ifelse(x, 1L, 0L)))
  if (is.character(x)) {
    xt <- trimws(x)
    if (xt %in% c("", "NA", "NaN")) return(NA_integer_)
    xn <- suppressWarnings(as.numeric(xt))
    return(ifelse(is.na(xn), NA_integer_, ifelse(xn == 1, 1L, 0L)))
  }
  xn <- suppressWarnings(as.numeric(x))
  ifelse(is.na(xn), NA_integer_, ifelse(xn == 1, 1L, 0L))
}

year_label <- function(y) {
  yn <- suppressWarnings(as.numeric(if (is.factor(y)) as.character(y) else y))
  ifelse(yn %in% c(0, 2010), "2010",
         ifelse(yn %in% c(1, 2019), "2019", NA_character_))
}

fmt_p3 <- function(p) {
  if (is.na(p)) return("NA")
  if (p < 0.001) return("<0.001")
  sprintf("%.3f", p)
}

fmt_n_pct <- function(n, denom) {
  if (is.na(n) || is.na(denom) || denom == 0) return("NA")
  sprintf("%d (%.1f%%)", n, 100 * n / denom)
}

fmt_median_iqr <- function(x, digits = 1) {
  x <- x[is.finite(x)]
  if (length(x) == 0) return("NA")
  q <- stats::quantile(x, probs = c(0.25, 0.5, 0.75), na.rm = TRUE, names = FALSE)
  sprintf(paste0("%.", digits, "f (%.", digits, "f-%.", digits, "f)"), q[2], q[1], q[3])
}

pval_wilcox <- function(x, g) {
  ok <- is.finite(x) & !is.na(g)
  x <- x[ok]; g <- g[ok]
  if (length(unique(g)) != 2) return(NA_real_)
  stats::wilcox.test(x ~ g, exact = FALSE)$p.value
}

pval_binary <- function(x01, g) {
  ok <- !is.na(x01) & !is.na(g)
  x01 <- x01[ok]; g <- g[ok]
  tab <- table(g, x01)
  if (nrow(tab) != 2 || ncol(tab) != 2) return(NA_real_)
  stats::fisher.test(tab)$p.value
}

pval_cat <- function(cat, g) {
  ok <- !is.na(cat) & !is.na(g)
  cat <- cat[ok]; g <- g[ok]
  tab <- table(g, cat)
  if (nrow(tab) < 2 || ncol(tab) < 2) return(NA_real_)
  exp <- suppressWarnings(stats::chisq.test(tab, correct = FALSE)$expected)
  if (any(exp < 5)) return(stats::chisq.test(tab, simulate.p.value = TRUE, B = 20000)$p.value)
  stats::chisq.test(tab, correct = FALSE)$p.value
}

# -------------------------
# Build published-only dataset
# -------------------------
df0$published01 <- as01(df0$Published)
df0$year_lab <- year_label(df0$Year)

pub <- df0[df0$published01 == 1 & df0$year_lab %in% c("2010","2019"), , drop = FALSE]

n_all  <- nrow(pub)
n_2010 <- sum(pub$year_lab == "2010")
n_2019 <- sum(pub$year_lab == "2019")

# numeric conversions
pub$time_to_pub <- suppressWarnings(as.numeric(pub$Timetopublicationmonthstimefromabstractapril2019))
pub$IF_raw      <- suppressWarnings(as.numeric(pub$Impactfactor))
pub$n_abs       <- suppressWarnings(as.numeric(pub$Numberofauthorsintheabstract))
pub$n_pub       <- suppressWarnings(as.numeric(pub$Numberofauthorsinthefinalpublication))
pub$delta_authors <- pub$n_pub - pub$n_abs

# Impact factor: treat nonpositive as missing
pub$IF <- ifelse(is.finite(pub$IF_raw) & pub$IF_raw > 0, pub$IF_raw, NA_real_)

# gender derivations (F=1, M=0, else NA)
pub$first_pub_female <- ifelse(toupper(trimws(as.character(pub$Firstauthorgenderofpublication))) == "F", 1L,
                               ifelse(toupper(trimws(as.character(pub$Firstauthorgenderofpublication))) == "M", 0L, NA_integer_))

pub$senior_pub_female <- ifelse(toupper(trimws(as.character(pub$Seniorauthorgenderofpublication))) == "F", 1L,
                                ifelse(toupper(trimws(as.character(pub$Seniorauthorgenderofpublication))) == "M", 0L, NA_integer_))

# author-count change category
pub$delta_cat <- NA_character_
pub$delta_cat[is.finite(pub$delta_authors) & pub$delta_authors > 0] <- "Increased"
pub$delta_cat[is.finite(pub$delta_authors) & pub$delta_authors == 0] <- "Unchanged"
pub$delta_cat[is.finite(pub$delta_authors) & pub$delta_authors < 0] <- "Decreased"

# split groups
g2010 <- pub[pub$year_lab == "2010", , drop = FALSE]
g2019 <- pub[pub$year_lab == "2019", , drop = FALSE]

# -------------------------
# Compute RAW p-values for the 7 primary comparisons
# (no IF >= 5 row)
# -------------------------
p_time   <- pval_wilcox(pub$time_to_pub, pub$year_lab)
p_if     <- pval_wilcox(pub$IF, pub$year_lab)
p_npub   <- pval_wilcox(pub$n_pub, pub$year_lab)
p_delta  <- pval_wilcox(pub$delta_authors, pub$year_lab)
p_dcat   <- pval_cat(pub$delta_cat, pub$year_lab)
p_ffirst <- pval_binary(pub$first_pub_female, pub$year_lab)
p_fsen   <- pval_binary(pub$senior_pub_female, pub$year_lab)

p_raw_vec <- c(
  time_to_pub = p_time,
  impact_factor = p_if,
  n_authors_final = p_npub,
  delta_authors = p_delta,
  delta_category = p_dcat,
  female_first = p_ffirst,
  female_senior = p_fsen
)

p_holm_vec <- rep(NA_real_, length(p_raw_vec))
okp <- is.finite(p_raw_vec)
p_holm_vec[okp] <- p.adjust(p_raw_vec[okp], method = "holm")
names(p_holm_vec) <- names(p_raw_vec)

# -------------------------
# Assemble Table 3
# -------------------------
col_overall <- sprintf("Overall published (n=%d)", n_all)
col_2010    <- sprintf("2010 published (n=%d)", n_2010)
col_2019    <- sprintf("2019 published (n=%d)", n_2019)

rows <- list()
add_row <- function(ch, o, y0, y1, p_raw, p_holm) {
  rows[[length(rows) + 1]] <<- data.frame(
    Characteristic = ch,
    o = o, y0 = y0, y1 = y1,
    p_raw = p_raw, p_holm = p_holm,
    stringsAsFactors = FALSE,
    check.names = FALSE
  )
}

# Time to publication
add_row(
  "Time to publication (months), median (IQR)",
  fmt_median_iqr(pub$time_to_pub, 1),
  fmt_median_iqr(g2010$time_to_pub, 1),
  fmt_median_iqr(g2019$time_to_pub, 1),
  fmt_p3(p_raw_vec["time_to_pub"]),
  fmt_p3(p_holm_vec["time_to_pub"])
)

# Impact factor (continuous only; thresholded row removed)
add_row(
  "Journal impact factor, median (IQR)",
  fmt_median_iqr(pub$IF, 1),
  fmt_median_iqr(g2010$IF, 1),
  fmt_median_iqr(g2019$IF, 1),
  fmt_p3(p_raw_vec["impact_factor"]),
  fmt_p3(p_holm_vec["impact_factor"])
)

# Final publication author count
add_row(
  "Number of authors in final publication, median (IQR)",
  fmt_median_iqr(pub$n_pub, 1),
  fmt_median_iqr(g2010$n_pub, 1),
  fmt_median_iqr(g2019$n_pub, 1),
  fmt_p3(p_raw_vec["n_authors_final"]),
  fmt_p3(p_holm_vec["n_authors_final"])
)

# Delta authors (publication - abstract)
add_row(
  "Change in author count (publication - abstract), median (IQR)",
  fmt_median_iqr(pub$delta_authors, 1),
  fmt_median_iqr(g2010$delta_authors, 1),
  fmt_median_iqr(g2019$delta_authors, 1),
  fmt_p3(p_raw_vec["delta_authors"]),
  fmt_p3(p_holm_vec["delta_authors"])
)

# Delta category distribution (global) + levels
add_row(
  "Change in author count category, n (%)",
  "-", "-", "-",
  fmt_p3(p_raw_vec["delta_category"]),
  fmt_p3(p_holm_vec["delta_category"])
)

delta_levels <- c("Decreased","Unchanged","Increased")
for (lvl in delta_levels) {
  add_row(
    paste0("  ", lvl),
    fmt_n_pct(sum(pub$delta_cat == lvl, na.rm = TRUE), sum(!is.na(pub$delta_cat))),
    fmt_n_pct(sum(g2010$delta_cat == lvl, na.rm = TRUE), sum(!is.na(g2010$delta_cat))),
    fmt_n_pct(sum(g2019$delta_cat == lvl, na.rm = TRUE), sum(!is.na(g2019$delta_cat))),
    "-", "-"
  )
}

# Female first author
add_row(
  "Female first author (publication), n (%)",
  fmt_n_pct(sum(pub$first_pub_female == 1, na.rm = TRUE), sum(!is.na(pub$first_pub_female))),
  fmt_n_pct(sum(g2010$first_pub_female == 1, na.rm = TRUE), sum(!is.na(g2010$first_pub_female))),
  fmt_n_pct(sum(g2019$first_pub_female == 1, na.rm = TRUE), sum(!is.na(g2019$first_pub_female))),
  fmt_p3(p_raw_vec["female_first"]),
  fmt_p3(p_holm_vec["female_first"])
)

# Female senior author
add_row(
  "Female senior author (publication), n (%)",
  fmt_n_pct(sum(pub$senior_pub_female == 1, na.rm = TRUE), sum(!is.na(pub$senior_pub_female))),
  fmt_n_pct(sum(g2010$senior_pub_female == 1, na.rm = TRUE), sum(!is.na(g2010$senior_pub_female))),
  fmt_n_pct(sum(g2019$senior_pub_female == 1, na.rm = TRUE), sum(!is.na(g2019$senior_pub_female))),
  fmt_p3(p_raw_vec["female_senior"]),
  fmt_p3(p_holm_vec["female_senior"])
)

table3 <- do.call(rbind, rows)

# Replace accidental blanks
for (j in seq_len(ncol(table3))) {
  table3[[j]][is.na(table3[[j]]) | table3[[j]] == ""] <- "NA"
}

names(table3) <- c("Characteristic", col_overall, col_2010, col_2019, "P-value (raw)", "P-value (Holm)")

# -------------------------
# Export CSV
# -------------------------
write.csv(table3, paste0(OUT_PREFIX, ".csv"), row.names = FALSE)

# -------------------------
# Export HTML (Word-openable)
# -------------------------
html_escape <- function(x) {
  x <- as.character(x)
  x <- gsub("&", "&amp;", x, fixed = TRUE)
  x <- gsub("<", "&lt;", x, fixed = TRUE)
  x <- gsub(">", "&gt;", x, fixed = TRUE)
  x
}

is_indent <- grepl("^\\s\\s", table3$Characteristic)
is_block  <- table3$Characteristic == "Change in author count category, n (%)"

css <- "
body { font-family: 'Times New Roman'; font-size: 10pt; }
table { border-collapse: collapse; width: 100%; }
caption { caption-side: top; text-align: left; font-weight: bold; margin-bottom: 6pt; }
th, td { padding: 4pt 6pt; vertical-align: top; }
th { text-align: left; border-bottom: 0.5pt solid #000; }
table { border-top: 1pt solid #000; border-bottom: 1pt solid #000; }
td.c { text-align: center; white-space: nowrap; }
td.r { text-align: right; white-space: nowrap; }
tr.hdr td { font-weight: bold; padding-top: 6pt; }
td.indent { padding-left: 14pt; }
.notes { margin-top: 8pt; font-size: 9pt; }
"

make_row <- function(i) {
  tr_class <- if (isTRUE(is_block[i])) " class='hdr'" else ""
  pred_class <- if (isTRUE(is_indent[i])) "indent" else ""

  ch <- html_escape(sub("^\\s\\s", "", table3[i, 1, drop = TRUE]))
  o  <- html_escape(table3[i, 2, drop = TRUE])
  y0 <- html_escape(table3[i, 3, drop = TRUE])
  y1 <- html_escape(table3[i, 4, drop = TRUE])
  pr <- html_escape(table3[i, 5, drop = TRUE])
  ph <- html_escape(table3[i, 6, drop = TRUE])

  sprintf(
    "<tr%s><td class='%s'>%s</td><td class='c'>%s</td><td class='c'>%s</td><td class='c'>%s</td><td class='r'>%s</td><td class='r'>%s</td></tr>",
    tr_class, pred_class, ch, o, y0, y1, pr, ph
  )
}

rows_html <- vapply(seq_len(nrow(table3)), make_row, character(1), USE.NAMES = FALSE)

caption <- sprintf(
  "Table 3. Characteristics of full-text publications arising from AANS podium abstracts (published only; 2010 n=%d, 2019 n=%d).",
  n_2010, n_2019
)

notes <- paste(
  "Notes: Values are n (%) or median (IQR).",
  "Raw p-values compare 2010 vs 2019 among published manuscripts (Fisher exact for binary; Wilcoxon rank-sum for continuous; chi-square with simulation for multi-level categorical).",
  "Holm-adjusted p-values control family-wise error across the 7 primary comparisons shown in the table (excluding indented subcategory rows).",
  "Impact factor values <= 0 were treated as missing.",
  "Gender variables reflect inferred/perceived gender and should be interpreted as a proxy rather than self-identified gender.",
  "Denominators may vary by row due to missingness in specific variables.",
  sep = " "
)

html <- c(
  "<html><head><meta charset='utf-8'/>",
  "<style>", css, "</style></head><body>",
  "<table>",
  "<caption>", html_escape(caption), "</caption>",
  "<tr>",
  "<th>Characteristic</th>",
  sprintf("<th>%s</th>", html_escape(col_overall)),
  sprintf("<th>%s</th>", html_escape(col_2010)),
  sprintf("<th>%s</th>", html_escape(col_2019)),
  "<th>P-value (raw)</th>",
  "<th>P-value (Holm)</th>",
  "</tr>",
  rows_html,
  "</table>",
  "<div class='notes'>", html_escape(notes), "</div>",
  "</body></html>"
)

writeLines(html, paste0(OUT_PREFIX, ".html"))

cat("Wrote:\n",
    "- ", OUT_PREFIX, ".csv\n",
    "- ", OUT_PREFIX, ".html (open in Word -> Save As .docx)\n", sep = "")

Wrote:
- /home/asegura/repos/aans-abstract-publication-2010-2019/outputs/table3_publication_characteristics_revised.csv
- /home/asegura/repos/aans-abstract-publication-2010-2019/outputs/table3_publication_characteristics_revised.html (open in Word -> Save As .docx)


## Figure 1
Kaplan–Meier curves for time to podium-to-print publication with administrative censoring at 36 months (plotted as 1 − S[t]).

In [31]:
# ============================================================
# Figure 1 (publication-ready, NO survminer):
# Kaplan–Meier time to podium-to-print publication (2010 vs 2019)
# - Right-censors unpublished abstracts at a fixed follow-up window
# - Plots cumulative probability published: 1 - S(t)
# - Wraps subtitle so text is never clipped
# - Adds numbers-at-risk table and ensures first/last cells aren’t clipped
#
# INPUT (uploaded):
#   <AANS_DATA_PATH>
#
# OUTPUT:
#   figure1_km_time_to_publication_professional.pdf
#   figure1_km_time_to_publication_professional.png
#   figure1_km_timepoint_estimates.csv
#   figure1_km_sessionInfo.txt
# ============================================================

# ---------- user knobs ----------
FOLLOW_UP_MONTHS <- 36          # fixed horizon for fair comparison
TIME_BREAK_BY    <- 6
SHOW_RISK_TABLE  <- TRUE

FIG_W_IN <- 8.5                 # wider to comfortably fit title/subtitle
FIG_H_IN <- if (SHOW_RISK_TABLE) 7 else 3.8
PNG_DPI  <- 600

BASE_SIZE <- 11                 # overall sizing; keep readable after reduction
TITLE_SIZE <- 16
SUBTITLE_SIZE <- 11

# ---------- packages (minimal) ----------
pkgs <- c("survival", "ggplot2", "scales", "patchwork")
need <- pkgs[!vapply(pkgs, requireNamespace, logical(1), quietly = TRUE)]
if (length(need)) install.packages(need, repos = "https://cloud.r-project.org")
invisible(lapply(pkgs, library, character.only = TRUE))
options(stringsAsFactors = FALSE)

# ---------- paths ----------
DATA_PATH <- AANS_DATA_PATH
stopifnot(file.exists(DATA_PATH))

df0 <- read.csv(DATA_PATH, check.names = FALSE)

# ---------- helpers ----------
as01 <- function(x) {
  if (is.factor(x)) x <- as.character(x)
  if (is.logical(x)) return(ifelse(is.na(x), NA_integer_, ifelse(x, 1L, 0L)))
  if (is.character(x)) {
    xt <- trimws(x)
    if (xt %in% c("", "NA", "NaN")) return(NA_integer_)
    xn <- suppressWarnings(as.numeric(xt))
    return(ifelse(is.na(xn), NA_integer_, ifelse(xn == 1, 1L, 0L)))
  }
  xn <- suppressWarnings(as.numeric(x))
  ifelse(is.na(xn), NA_integer_, ifelse(xn == 1, 1L, 0L))
}

year_label <- function(y) {
  yn <- suppressWarnings(as.numeric(if (is.factor(y)) as.character(y) else y))
  ifelse(yn %in% c(0, 2010), "2010",
         ifelse(yn %in% c(1, 2019), "2019", NA_character_))
}

fmt_p <- function(p) {
  if (is.na(p)) return("NA")
  if (p < 0.001) return("<0.001")
  sprintf("%.3f", p)
}

wrap_text <- function(x, width = 95) paste(strwrap(x, width = width), collapse = "\n")

# Build tidy curve data from survfit (step function)
survfit_to_df <- function(fit, t_end) {
  strata_names <- names(fit$strata)
  counts <- as.integer(fit$strata)

  ends <- cumsum(counts)
  starts <- c(1, head(ends, -1) + 1)

  out <- vector("list", length(strata_names))
  for (i in seq_along(strata_names)) {
    idx <- starts[i]:ends[i]
    ti <- fit$time[idx]
    si <- fit$surv[idx]

    df_i <- data.frame(time = c(0, ti), surv = c(1, si), strata = strata_names[i])

    # Extend flat to t_end so curves align with the same x-limit
    last_t <- tail(df_i$time, 1)
    last_s <- tail(df_i$surv, 1)
    if (is.finite(t_end) && last_t < t_end) {
      df_i <- rbind(df_i, data.frame(time = t_end, surv = last_s, strata = strata_names[i]))
    }
    out[[i]] <- df_i
  }
  do.call(rbind, out)
}

# ---------- required columns ----------
req <- c("Year", "Published", "Timetopublicationmonthstimefromabstractapril2019")
stopifnot(all(req %in% names(df0)))

# ---------- build analysis dataset ----------
year <- year_label(df0$Year)
published01 <- as01(df0$Published)
t_pub <- suppressWarnings(as.numeric(df0$Timetopublicationmonthstimefromabstractapril2019))

keep <- !is.na(year) & year %in% c("2010", "2019")
dat <- data.frame(
  year = factor(year[keep], levels = c("2010", "2019")),
  published01 = published01[keep],
  t_pub = t_pub[keep]
)

# Drop rows that claim published but have missing/invalid time-to-publication
bad <- which(dat$published01 == 1 & !is.finite(dat$t_pub))
if (length(bad)) {
  warning(sprintf("Dropping %d row(s): Published==1 but missing/invalid time-to-publication.", length(bad)))
  dat <- dat[-bad, , drop = FALSE]
}

# Administrative censoring for fair comparison
t_end <- FOLLOW_UP_MONTHS
dat$time <- ifelse(dat$published01 == 1 & is.finite(dat$t_pub),
                   pmin(dat$t_pub, t_end),
                   t_end)
dat$event <- as.integer(dat$published01 == 1 & is.finite(dat$t_pub) & dat$t_pub <= t_end)

# ---------- survival fit + log-rank ----------
surv_obj <- survival::Surv(time = dat$time, event = dat$event)
fit <- survival::survfit(surv_obj ~ year, data = dat)

lr <- survival::survdiff(surv_obj ~ year, data = dat)
p_lr <- 1 - pchisq(lr$chisq, df = length(lr$n) - 1)

n_total <- table(dat$year)
n_events <- tapply(dat$event, dat$year, sum)

subtitle_raw <- paste0(
  "Log-rank p=", fmt_p(p_lr),
  " | 2010: ", n_events[["2010"]], "/", n_total[["2010"]], " published within window; ",
  "2019: ", n_events[["2019"]], "/", n_total[["2019"]], " | ",
  "Censored at ", t_end, " months"
)
subtitle_txt <- wrap_text(subtitle_raw, width = 92)

# ---------- curve dataframe (plot 1 - S(t)) ----------
df_curve <- survfit_to_df(fit, t_end)
df_curve$year <- factor(sub("^year=", "", df_curve$strata), levels = c("2010", "2019"))
df_curve$cum_published <- 1 - df_curve$surv

# ---------- numbers at risk ----------
times_risk <- seq(0, t_end, by = TIME_BREAK_BY)
sm <- summary(fit, times = times_risk, extend = TRUE)
df_risk <- data.frame(
  year = factor(sub("^year=", "", sm$strata), levels = c("2010", "2019")),
  time = sm$time,
  n_risk = sm$n.risk
)

# Avoid clipping at the first/last cell by adding x padding and adjusting hjust at time 0
df_risk$hjust <- ifelse(df_risk$time == 0, 0, 0.5)

# Shared x-scale padding (in months) so labels at 0 and 36 don’t clip
x_pad_left  <- 0.9
x_pad_right <- 0.9

x_scale <- scale_x_continuous(
  breaks = times_risk,
  limits = c(0, t_end),
  expand = expansion(add = c(x_pad_left, x_pad_right))
)

# ---------- plot (top) ----------
p_curve <- ggplot(df_curve, aes(x = time, y = cum_published, linetype = year, color = year)) +
  geom_step(direction = "hv", linewidth = 0.9) +
  x_scale +
  scale_y_continuous(
    labels = scales::label_percent(accuracy = 1),
    limits = c(0, 1),
    breaks = scales::pretty_breaks(5)
  ) +
  scale_color_manual(values = c("2010" = "black", "2019" = "gray40")) +
  scale_linetype_manual(values = c("2010" = "solid", "2019" = "dashed")) +
  labs(
    title = "Time to podium-to-print publication",
#     subtitle = subtitle_txt,
    x = NULL,
    y = "Cumulative probability published",
    color = NULL,
    linetype = NULL
  ) +
  theme_classic(base_size = BASE_SIZE) +
  theme(
    text = element_text(family = "sans"),
    plot.title = element_text(face = "bold", size = TITLE_SIZE, hjust = 0.5),
#    plot.subtitle = element_text(size = SUBTITLE_SIZE, hjust = 0.5, lineheight = 1.05),
    legend.position = "top",
    legend.text = element_text(size = 12),
    axis.text.x = element_blank(),
    axis.ticks.x = element_blank(),
    plot.margin = margin(t = 12, r = 28, b = 6, l = 18)
  )

# ---------- risk table (bottom) ----------
p_risk <- ggplot(df_risk, aes(x = time, y = year)) +
  geom_text(aes(label = n_risk, hjust = hjust), size = 4) +
  x_scale +
  labs(title = "Number at risk", x = "Months from AANS podium presentation", y = NULL) +
  theme_classic(base_size = BASE_SIZE) +
  theme(
    text = element_text(family = "sans"),
    plot.title = element_text(face = "bold", size = 12, hjust = 0),
    axis.line.y = element_blank(),
    axis.ticks.y = element_blank(),
    plot.margin = margin(t = 0, r = 28, b = 10, l = 18)
  )

# ---------- combine ----------
if (SHOW_RISK_TABLE) {
  fig <- p_curve / p_risk + patchwork::plot_layout(heights = c(3.2, 1.1))
} else {
  fig <- p_curve + theme(axis.text.x = element_text(), axis.ticks.x = element_line()) +
    labs(x = "Months from AANS podium presentation")
}

# ---------- export ----------
pdf_file <- file.path(AANS_OUT_DIR, "figure1_km_time_to_publication_professional.pdf")
png_file <- file.path(AANS_OUT_DIR, "figure1_km_time_to_publication_professional.png")

if (capabilities("cairo")) {
  ggsave(pdf_file, fig, width = FIG_W_IN, height = FIG_H_IN, units = "in", device = cairo_pdf)
} else {
  ggsave(pdf_file, fig, width = FIG_W_IN, height = FIG_H_IN, units = "in")
}
ggsave(png_file, fig, width = FIG_W_IN, height = FIG_H_IN, units = "in", dpi = PNG_DPI)

# ---------- timepoint estimates (optional, useful for Results text) ----------
times_out <- c(6, 12, 24, 36)
times_out <- times_out[times_out <= t_end]
sm2 <- summary(fit, times = times_out, extend = TRUE)
tp <- data.frame(
  year = sub("^year=", "", sm2$strata),
  months = sm2$time,
  surv_not_published = sm2$surv,
  cum_published = 1 - sm2$surv,
  cum_published_lcl = 1 - sm2$upper,
  cum_published_ucl = 1 - sm2$lower,
  stringsAsFactors = FALSE
)
write.csv(tp, file.path(AANS_OUT_DIR, "figure1_km_timepoint_estimates.csv"), row.names = FALSE)

writeLines(capture.output(sessionInfo()), file.path(AANS_OUT_DIR, "figure1_km_sessionInfo.txt"))

cat("Wrote:\n- ", pdf_file,
    "\n- ", png_file,
    "\n- figure1_km_timepoint_estimates.csv\n- figure1_km_sessionInfo.txt\n", sep = "")

Warning message:
“Dropping 1 row(s): Published==1 but missing/invalid time-to-publication.”


Wrote:
- /home/asegura/repos/aans-abstract-publication-2010-2019/outputs/figure1_km_time_to_publication_professional.pdf
- /home/asegura/repos/aans-abstract-publication-2010-2019/outputs/figure1_km_time_to_publication_professional.png
- figure1_km_timepoint_estimates.csv
- figure1_km_sessionInfo.txt


## Figure 2
Two-panel plot: medical-student first authorship increased and publication within 36 months declined.

In [32]:
# ============================================================
# Figure 2 (2-panel): Student first authorship ↑ + Publication ↓
# FIXES:
#   - Panel B endpoint truly reflects "Published within 36 months"
#   - Removes .data pronoun usage outside dplyr masks (fixes your error)
#   - Uses explicit 'outcome' column for fisher tests and glm
#   - Adds caption note if published abstracts have missing time_to_pub
#
# Output:
#   figure2_clean_main.png
#   figure2_clean_main.pdf
# ============================================================

DATA_PATH <- AANS_DATA_PATH

OUT_PREFIX <- file.path(AANS_OUT_DIR, "figure2_clean_main")
DPI <- 300
WIDTH_IN <- 8.0
HEIGHT_IN <- 6.8

# If TRUE, prints p-values as a bottom caption (recommended).
SHOW_PVAL_CAPTION <- TRUE

# ---- Endpoint control ----
USE_PUBLICATION_WINDOW <- TRUE
PUB_WINDOW_MONTHS <- 36

# If NULL, auto-detect a time-to-publication column.
# If detection is wrong, set this manually to the exact column name in your CSV.
TIME_TO_PUB_COL <- NULL

# How to handle Published==1 but missing time-to-publication:
#   "drop" = exclude from within-window endpoint (default; matches your KM denominator 135 in 2010)
#   "assume_over" = treat as NOT within window (0) to keep denominators
#   "assume_within" = treat as within window (1) (rarely defensible)
MISSING_TIME_PUBLISHED_POLICY <- "drop"
# --------------------------

options(stringsAsFactors = FALSE)
options(repos = c(CRAN = "https://cloud.r-project.org"))

install_if_missing <- function(pkgs) {
  for (p in pkgs) {
    if (!requireNamespace(p, quietly = TRUE)) {
      try(install.packages(p, quiet = TRUE), silent = TRUE)
    }
  }
}

install_if_missing(c("dplyr", "ggplot2", "scales", "patchwork"))

library(dplyr)
library(ggplot2)
library(scales)
library(patchwork)

# ---------- locate data ----------
stopifnot(file.exists(DATA_PATH))

df0 <- read.csv(DATA_PATH, check.names = FALSE)

required_cols <- c("Year", "Medstudentfirstauthor1Yes0No", "Published")
missing_cols <- setdiff(required_cols, names(df0))
if (length(missing_cols) > 0) {
  stop(paste0("Missing required columns:\n- ", paste(missing_cols, collapse = "\n- ")))
}

# ---------- helpers ----------
as01 <- function(x) {
  if (is.factor(x)) x <- as.character(x)
  if (is.logical(x)) return(ifelse(is.na(x), NA_integer_, ifelse(x, 1L, 0L)))
  if (is.character(x)) {
    xt <- trimws(x)
    if (xt %in% c("", "NA", "NaN")) return(NA_integer_)
    xn <- suppressWarnings(as.numeric(xt))
    return(ifelse(is.na(xn), NA_integer_, ifelse(xn == 1, 1L, 0L)))
  }
  xn <- suppressWarnings(as.numeric(x))
  ifelse(is.na(xn), NA_integer_, ifelse(xn == 1, 1L, 0L))
}

clean_year <- function(y) {
  yn <- suppressWarnings(as.numeric(y))
  if (all(na.omit(yn) %in% c(0, 1))) {
    ifelse(yn == 0, "2010", ifelse(yn == 1, "2019", NA_character_))
  } else if (all(na.omit(yn) %in% c(2010, 2019))) {
    as.character(yn)
  } else {
    as.character(y)
  }
}

fmt_p <- function(p) {
  if (is.na(p)) return("-")
  if (p < 0.001) return("<0.001")
  sprintf("%.3f", p)
}

exact_ci <- function(x, n, conf.level = 0.95) {
  if (is.na(x) || is.na(n) || n <= 0) return(c(NA_real_, NA_real_))
  bt <- stats::binom.test(x = x, n = n, conf.level = conf.level)
  c(bt$conf.int[1], bt$conf.int[2])
}

to_numeric_loose <- function(x) {
  if (is.factor(x)) x <- as.character(x)
  if (is.character(x)) {
    x <- gsub(",", "", x)
    x <- gsub("[^0-9.]+", "", x)
  }
  suppressWarnings(as.numeric(x))
}

# ---------- auto-detect time-to-publication column ----------
use_window_final <- FALSE
if (USE_PUBLICATION_WINDOW) {
  if (is.null(TIME_TO_PUB_COL)) {
    nm <- names(df0)
    nm_low <- tolower(nm)
    hits <- which(
      grepl("time.*publication", nm_low) |
        grepl("publication.*time", nm_low) |
        grepl("time_to_publication", nm_low) |
        grepl("months.*publication", nm_low) |
        grepl("time.*to.*pub", nm_low) |
        grepl("podium.*print.*time", nm_low)
    )
    if (length(hits) > 0) {
      TIME_TO_PUB_COL <- nm[hits[1]]
      message("Detected TIME_TO_PUB_COL = '", TIME_TO_PUB_COL, "'")
      use_window_final <- TRUE
    } else {
      message("No time-to-publication column detected. Panel B will fall back to overall publication and relabel.")
      use_window_final <- FALSE
    }
  } else {
    stopifnot(TIME_TO_PUB_COL %in% names(df0))
    message("Using TIME_TO_PUB_COL = '", TIME_TO_PUB_COL, "'")
    use_window_final <- TRUE
  }
}

# ---------- recode base ----------
df <- df0 %>%
  mutate(
    Year_lab = clean_year(Year),
    Year_num = suppressWarnings(as.integer(clean_year(Year))),
    student01 = as01(Medstudentfirstauthor1Yes0No),
    published01 = as01(Published),
    time_to_pub_mo = if (use_window_final) to_numeric_loose(.data[[TIME_TO_PUB_COL]]) else NA_real_
  ) %>%
  filter(Year_lab %in% c("2010", "2019")) %>%
  filter(!is.na(student01)) %>%
  mutate(
    Student_lab = ifelse(student01 == 1, "Medical student", "Non-student"),
    Student_lab = factor(Student_lab, levels = c("Non-student", "Medical student"))
  )

# ---------- build Panel B endpoint ----------
if (use_window_final) {
  # handle published==1 but missing time according to policy
  dfB <- df %>%
    filter(!is.na(published01)) %>%
    mutate(
      pub_within_window = case_when(
        published01 == 0 ~ 0L,
        published01 == 1 & !is.na(time_to_pub_mo) ~ ifelse(time_to_pub_mo <= PUB_WINDOW_MONTHS, 1L, 0L),
        published01 == 1 & is.na(time_to_pub_mo) & MISSING_TIME_PUBLISHED_POLICY == "assume_over" ~ 0L,
        published01 == 1 & is.na(time_to_pub_mo) & MISSING_TIME_PUBLISHED_POLICY == "assume_within" ~ 1L,
        TRUE ~ NA_integer_
      )
    )

  if (MISSING_TIME_PUBLISHED_POLICY == "drop") {
    dfB <- dfB %>% filter(!(published01 == 1 & is.na(time_to_pub_mo)))
  } else {
    dfB <- dfB %>% filter(!is.na(pub_within_window))
  }

  outcome_title <- paste0("B. Publication within ", PUB_WINDOW_MONTHS, " months declined in both strata")
  outcome_ylabel <- paste0("Published within ", PUB_WINDOW_MONTHS, " months")
  outcome_vec <- dfB$pub_within_window
} else {
  dfB <- df %>% filter(!is.na(published01)) %>% mutate(pub_within_window = published01)
  outcome_title <- "B. Publication declined in both strata"
  outcome_ylabel <- "Published"
  outcome_vec <- dfB$pub_within_window
}

# Panel A can remain full cohort (student share) to match Table 1 denominators.
dfA <- df %>% filter(!is.na(Year_lab))

# ---------- sanity prints ----------
message("\nSanity check: totals by year (Panel A denominator):\n")
print(dfA %>% count(Year_lab))

message("\nSanity check: totals by year and student stratum (Panel B denominator):\n")
print(dfB %>% count(Year_lab, Student_lab))

# Missing time-to-publication among published
if (use_window_final) {
  miss_tbl <- df %>%
    filter(published01 == 1 & is.na(time_to_pub_mo)) %>%
    count(Year_lab, Student_lab, name = "n_missing_time")
  n_missing_time_published <- sum(miss_tbl$n_missing_time)
  if (n_missing_time_published > 0) {
    message("\nNOTE: Published abstracts with missing time_to_pub_mo:\n")
    print(miss_tbl)
  }
}

# ---------- Panel A: student share ----------
sum_student <- dfA %>%
  group_by(Year_num) %>%
  summarise(
    n = n(),
    student_n = sum(student01 == 1),
    prop = student_n / n,
    .groups = "drop"
  )

ciA <- mapply(exact_ci, x = sum_student$student_n, n = sum_student$n)
sum_student$ci_low  <- as.numeric(ciA[1, ])
sum_student$ci_high <- as.numeric(ciA[2, ])
sum_student$label <- sprintf("%d/%d\n(%.1f%%)", sum_student$student_n, sum_student$n, 100 * sum_student$prop)

# ---------- Panel B: publication share by stratum ----------
sum_pub <- dfB %>%
  group_by(Year_num, Student_lab) %>%
  summarise(
    n = n(),
    pub_n = sum(pub_within_window == 1),
    prop = pub_n / n,
    .groups = "drop"
  )

ciB <- mapply(exact_ci, x = sum_pub$pub_n, n = sum_pub$n)
sum_pub$ci_low  <- as.numeric(ciB[1, ])
sum_pub$ci_high <- as.numeric(ciB[2, ])
sum_pub$label <- sprintf("%d/%d\n(%.1f%%)", sum_pub$pub_n, sum_pub$n, 100 * sum_pub$prop)

# ---------- tests for caption ----------
tab_student_share <- table(dfA$student01, dfA$Year_lab)
p_student_share <- stats::fisher.test(tab_student_share)$p.value

tab_pub_overall <- table(dfB$pub_within_window, dfB$Year_lab)
p_pub_overall <- stats::fisher.test(tab_pub_overall)$p.value

df_lr <- dfB %>%
  mutate(
    year01 = ifelse(Year_lab == "2019", 1, 0),
    student_bin = ifelse(Student_lab == "Medical student", 1, 0),
    outcome = pub_within_window
  )

m0 <- glm(outcome ~ year01 + student_bin, data = df_lr, family = binomial())
m1 <- glm(outcome ~ year01 * student_bin, data = df_lr, family = binomial())
p_interaction <- anova(m0, m1, test = "LRT")$`Pr(>Chi)`[2]

note_txt <- ""
if (use_window_final) {
  n_missing_time_published <- sum(df$published01 == 1 & is.na(df$time_to_pub_mo), na.rm = TRUE)
  if (n_missing_time_published > 0 && MISSING_TIME_PUBLISHED_POLICY == "drop") {
    note_txt <- paste0(" Note: ", n_missing_time_published,
                       " published abstract(s) had missing time-to-publication and were excluded from the within-window endpoint.")
  }
}

caption_txt <- paste0(
  "P-values: student-share p=", fmt_p(p_student_share),
  "; publication p=", fmt_p(p_pub_overall),
  "; interaction p=", fmt_p(p_interaction), " (LRT). ",
  "Error bars are exact 95% binomial CIs.",
  note_txt
)

# ---------- common theme tweaks ----------
base_theme <- theme_classic(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    axis.title.y = element_text(margin = margin(r = 10)),
    plot.margin = margin(10, 18, 10, 18)
  )

# ---------- Panel A plot ----------
yA_max <- min(1.0, max(sum_student$ci_high, na.rm = TRUE) + 0.10)

pA <- ggplot(sum_student, aes(x = Year_num, y = prop)) +
  geom_errorbar(aes(ymin = ci_low, ymax = ci_high), width = 0.0, linewidth = 0.8) +
  geom_point(size = 3, shape = 17) +
  geom_text(
    data = subset(sum_student, Year_num == 2010),
    aes(label = label),
    nudge_x = 0.8, hjust = 0, vjust = -0.6, size = 3.2
  ) +
  geom_text(
    data = subset(sum_student, Year_num == 2019),
    aes(label = label),
    nudge_x = -0.8, hjust = 1, vjust = -0.6, size = 3.2
  ) +
  scale_x_continuous(
    breaks = c(2010, 2019),
    labels = c("2010", "2019"),
    expand = expansion(mult = c(0.20, 0.20))
  ) +
  scale_y_continuous(
    labels = percent_format(accuracy = 1),
    limits = c(0, yA_max),
    expand = expansion(mult = c(0.00, 0.02))
  ) +
  coord_cartesian(clip = "off") +
  labs(
    title = "A. Medical-student first authorship increased",
    y = "Medical-student first author",
    x = "Year"
  ) +
  base_theme

# ---------- Panel B plot ----------
yB_max <- min(1.0, max(sum_pub$ci_high, na.rm = TRUE) + 0.10)

pB <- ggplot(sum_pub, aes(x = Year_num, y = prop)) +
  geom_errorbar(aes(ymin = ci_low, ymax = ci_high), width = 0.0, linewidth = 0.8) +
  geom_point(size = 3) +
  geom_line(aes(group = 1), linewidth = 0.6) +
  geom_text(
    data = subset(sum_pub, Year_num == 2010),
    aes(label = label),
    nudge_x = 0.8, hjust = 0, vjust = -0.6, size = 3.0
  ) +
  geom_text(
    data = subset(sum_pub, Year_num == 2019),
    aes(label = label),
    nudge_x = -0.8, hjust = 1, vjust = -0.6, size = 3.0
  ) +
  facet_wrap(~ Student_lab, nrow = 1) +
  scale_x_continuous(
    breaks = c(2010, 2019),
    labels = c("2010", "2019"),
    expand = expansion(mult = c(0.20, 0.20))
  ) +
  scale_y_continuous(
    labels = percent_format(accuracy = 1),
    limits = c(0, yB_max),
    expand = expansion(mult = c(0.00, 0.02))
  ) +
  coord_cartesian(clip = "off") +
  labs(
    title = outcome_title,
    y = outcome_ylabel,
    x = "Year"
  ) +
  base_theme +
  theme(
    strip.background = element_blank(),
    strip.text = element_text(face = "bold"),
    panel.spacing = grid::unit(2.2, "lines")
  )

# ---------- Combine with patchwork ----------
main_title <- "Podium-to-print publication declined\ndespite increased medical-student first authorship"

fig <- pA / pB +
  plot_layout(heights = c(1, 1.2)) +
  plot_annotation(
    title = main_title,
    caption = if (SHOW_PVAL_CAPTION) caption_txt else NULL,
    theme = theme(
      plot.title = element_text(face = "bold", size = 18),
      plot.caption = element_text(size = 10, hjust = 0),
      plot.margin = margin(12, 18, 12, 18)
    )
  )

ggsave(paste0(OUT_PREFIX, ".png"), fig, width = WIDTH_IN, height = HEIGHT_IN, dpi = DPI)
ggsave(paste0(OUT_PREFIX, ".pdf"), fig, width = WIDTH_IN, height = HEIGHT_IN)

message("Wrote:\n- ", OUT_PREFIX, ".png\n- ", OUT_PREFIX, ".pdf")

Detected TIME_TO_PUB_COL = 'Timetopublicationmonthstimefromabstractapril2019'


Sanity check: totals by year (Panel A denominator):




  Year_lab   n
1     2010 136
2     2019 217



Sanity check: totals by year and student stratum (Panel B denominator):




  Year_lab     Student_lab   n
1     2010     Non-student 126
2     2010 Medical student   9
3     2019     Non-student 114
4     2019 Medical student 103



NOTE: Published abstracts with missing time_to_pub_mo:




  Year_lab Student_lab n_missing_time
1     2010 Non-student              1


Wrote:
- /home/asegura/repos/aans-abstract-publication-2010-2019/outputs/figure2_clean_main.png
- /home/asegura/repos/aans-abstract-publication-2010-2019/outputs/figure2_clean_main.pdf



## Optional: Forest plot of adjusted ORs
A visual companion to Table 2 (can be used as a supplemental figure).

In [33]:
#!/usr/bin/env Rscript

# ============================================================
# Figure 2 (Manuscript): Forest plot of adjusted ORs
# - Multivariable logistic regression (same covariates as Table 2)
# - Complete-case dataset for stable N
# - Two-panel layout: forest plot (left) + aligned text table (right)
# - FIX: ensure both panels use the SAME rows and SAME y-scale limits
#
# INPUT:
#   <AANS_DATA_PATH>
#
# OUTPUT:
#   supplemental_figure_adjusted_OR_forestplot_clean_aligned.png
#   supplemental_figure_adjusted_OR_forestplot_clean_aligned.pdf
#   supplemental_figure_adjusted_OR_forestplot_clean_aligned_table.csv
# ============================================================

DATA_PATH <- AANS_DATA_PATH
OUT_PREFIX <- file.path(AANS_OUT_DIR, "supplemental_figure_adjusted_OR_forestplot_clean_aligned")

stopifnot(file.exists(DATA_PATH))

options(stringsAsFactors = FALSE)

suppressPackageStartupMessages({
  pkgs <- c("dplyr", "ggplot2", "stringr", "janitor", "readr", "patchwork", "fs")
  missing <- pkgs[!vapply(pkgs, requireNamespace, logical(1), quietly = TRUE)]
  if (length(missing) > 0) install.packages(missing, repos = "https://cloud.r-project.org")
  invisible(lapply(pkgs, library, character.only = TRUE))
})

# -------------------------
# Helpers
# -------------------------
as01 <- function(x) {
  if (is.factor(x)) x <- as.character(x)
  if (is.logical(x)) return(ifelse(is.na(x), NA_integer_, ifelse(x, 1L, 0L)))
  if (is.character(x)) {
    xt <- trimws(x)
    if (xt %in% c("", "NA", "NaN")) return(NA_integer_)
    xn <- suppressWarnings(as.numeric(xt))
    return(ifelse(is.na(xn), NA_integer_, ifelse(xn == 1, 1L, 0L)))
  }
  xn <- suppressWarnings(as.numeric(x))
  ifelse(is.na(xn), NA_integer_, ifelse(xn == 1, 1L, 0L))
}

clean_year <- function(y) {
  yn <- suppressWarnings(as.numeric(y))
  if (all(na.omit(yn) %in% c(0, 1))) {
    ifelse(yn == 0, "2010", ifelse(yn == 1, "2019", NA_character_))
  } else if (all(na.omit(yn) %in% c(2010, 2019))) {
    as.character(yn)
  } else {
    as.character(y)
  }
}

fmt_p3 <- function(p) {
  if (!is.finite(p) || is.na(p)) return("NA")
  if (p < 0.001) return("<0.001")
  sprintf("%.3f", p)
}

fmt_or_ci <- function(or, lcl, ucl, digits = 2) {
  if (!is.finite(or) || !is.finite(lcl) || !is.finite(ucl)) return("NA")
  sprintf(paste0("%.", digits, "f (%.", digits, "f-%.", digits, "f)"), or, lcl, ucl)
}

# -------------------------
# Load data + verify columns
# -------------------------
df0 <- read.csv(DATA_PATH, check.names = FALSE) %>% janitor::clean_names()

required_cols <- c(
  "year",
  "subspecialty",
  "medstudentfirstauthor1yes0no",
  "seniorauthorcountry",
  "numberofauthorsintheabstract",
  "firstauthorgenderabstract_mf",
  "seniorauthorgenderofabstract_mf",
  "award",
  "published"
)
missing_cols <- setdiff(required_cols, names(df0))
if (length(missing_cols) > 0) {
  stop(paste0("Missing columns after clean_names():\n- ", paste(missing_cols, collapse = "\n- ")))
}

# -------------------------
# Recode for modeling (stable coefficient names)
# -------------------------
df <- df0
df$year_f <- factor(clean_year(df$year), levels = c("2010", "2019"))

sub_map <- c(
  "1" = "Functional/Peripheral Nerve",
  "2" = "Pediatrics",
  "3" = "Spine",
  "4" = "Vascular/Stroke",
  "5" = "General/Other",
  "6" = "Trauma/Critical Care",
  "7" = "Tumor"
)
sub_raw <- trimws(as.character(df$subspecialty))
sub_num <- suppressWarnings(as.numeric(sub_raw))
sub_lab <- sub_raw
if (all(na.omit(sub_num) %in% suppressWarnings(as.numeric(names(sub_map))))) {
  sub_lab <- unname(sub_map[as.character(sub_num)])
}

sub_code_map <- c(
  "General/Other" = "general_other",
  "Functional/Peripheral Nerve" = "functional_peripheral_nerve",
  "Pediatrics" = "pediatrics",
  "Spine" = "spine",
  "Trauma/Critical Care" = "trauma_critical_care",
  "Tumor" = "tumor",
  "Vascular/Stroke" = "vascular_stroke"
)

df$subspec <- unname(sub_code_map[sub_lab])
df$subspec <- factor(
  df$subspec,
  levels = c("general_other",
             "functional_peripheral_nerve",
             "pediatrics",
             "spine",
             "trauma_critical_care",
             "tumor",
             "vascular_stroke")
)

df$med_student <- factor(ifelse(as01(df$medstudentfirstauthor1yes0no) == 1, "yes", "no"),
                         levels = c("no", "yes"))

df$senior_country <- factor(ifelse(as01(df$seniorauthorcountry) == 1, "international", "usa"),
                            levels = c("usa", "international"))

df$n_authors <- suppressWarnings(as.numeric(df$numberofauthorsintheabstract))

df$first_gender <- factor(ifelse(as01(df$firstauthorgenderabstract_mf) == 1, "female", "male"),
                          levels = c("male", "female"))
df$senior_gender <- factor(ifelse(as01(df$seniorauthorgenderofabstract_mf) == 1, "female", "male"),
                           levels = c("male", "female"))

df$award_f <- factor(ifelse(as01(df$award) == 1, "yes", "no"),
                     levels = c("no", "yes"))

df$published01 <- as01(df$published)

# Keep only 2010/2019 + non-missing outcome
df <- df[!is.na(df$year_f) & df$year_f %in% c("2010", "2019") & !is.na(df$published01), , drop = FALSE]

# Complete-case dataset
vars_needed <- c("published01","year_f","subspec","med_student","senior_country","n_authors","first_gender","senior_gender","award_f")
mdata <- df[, vars_needed, drop = FALSE]
mdata <- mdata[complete.cases(mdata), , drop = FALSE]

N_model <- nrow(mdata)
events  <- sum(mdata$published01 == 1)

# -------------------------
# Fit multivariable model
# -------------------------
fit_adj <- glm(
  published01 ~ year_f + subspec + med_student + senior_country + n_authors + first_gender + senior_gender + award_f,
  family = binomial(),
  data = mdata
)

co <- summary(fit_adj)$coefficients
co_df <- as.data.frame(co)
co_df$term <- rownames(co_df)
rownames(co_df) <- NULL

co_df <- co_df[co_df$term != "(Intercept)", , drop = FALSE] %>%
  dplyr::mutate(
    beta = Estimate,
    se = `Std. Error`,
    p = `Pr(>|z|)`,
    or = exp(beta),
    lcl = exp(beta - 1.96 * se),
    ucl = exp(beta + 1.96 * se),
    orci_text = mapply(fmt_or_ci, or, lcl, ucl, MoreArgs = list(digits = 2)),
    p_text = vapply(p, fmt_p3, character(1))
  )

# -------------------------
# Labels + ordering (NO header rows)
# -------------------------
label_map <- c(
  "year_f2019" = "Year: 2019 vs 2010",
  "subspecfunctional_peripheral_nerve" = "Functional/Peripheral Nerve vs General/Other",
  "subspecpediatrics" = "Pediatrics vs General/Other",
  "subspecspine" = "Spine vs General/Other",
  "subspectrauma_critical_care" = "Trauma/Critical Care vs General/Other",
  "subspectumor" = "Tumor vs General/Other",
  "subspecvascular_stroke" = "Vascular/Stroke vs General/Other",
  "med_studentyes" = "Medical student first author: Yes vs No",
  "senior_countryinternational" = "International senior author: International vs USA",
  "n_authors" = "Number of authors in abstract (per +1 author)",
  "first_genderfemale" = "Female first author (abstract): Female vs Male",
  "senior_genderfemale" = "Female senior author (abstract): Female vs Male",
  "award_fyes" = "Award-winning abstract: Yes vs No"
)

present_terms <- co_df$term
sub_terms_all <- c(
  "subspecfunctional_peripheral_nerve",
  "subspecpediatrics",
  "subspecspine",
  "subspectrauma_critical_care",
  "subspectumor",
  "subspecvascular_stroke"
)
sub_terms_present <- intersect(sub_terms_all, present_terms)

# Desired top-to-bottom order in the figure:
row_order <- c(
  "year_f2019",
  sub_terms_present,
  "med_studentyes",
  "senior_countryinternational",
  "n_authors",
  "first_genderfemale",
  "senior_genderfemale",
  "award_fyes"
)
row_order <- row_order[row_order %in% present_terms]  # safety

plot_df <- co_df %>%
  dplyr::filter(term %in% row_order) %>%
  dplyr::mutate(label = dplyr::if_else(term %in% names(label_map), unname(label_map[term]), term)) %>%
  dplyr::select(term, label, or, lcl, ucl, orci_text, p_text)

# IMPORTANT: Use identical y-levels for both panels
label_levels_top_to_bottom <- unname(label_map[row_order])
label_levels_top_to_bottom <- label_levels_top_to_bottom[!is.na(label_levels_top_to_bottom) & nzchar(label_levels_top_to_bottom)]

# ggplot draws first factor level at bottom, so reverse to get top-to-bottom ordering
plot_df$label <- factor(plot_df$label, levels = rev(label_levels_top_to_bottom))

# -------------------------
# X-scale limits for forest plot (data only)
# -------------------------
x_data_max <- max(plot_df$ucl, na.rm = TRUE)
x_data_min <- min(plot_df$lcl[is.finite(plot_df$lcl) & plot_df$lcl > 0], na.rm = TRUE)

x_left  <- max(0.2, x_data_min * 0.80)
x_right <- max(8, x_data_max * 1.25)

base_breaks <- c(0.25, 0.5, 1, 2, 4, 8, 16, 32)
x_breaks <- base_breaks[base_breaks >= x_left & base_breaks <= x_right]
if (!any(abs(x_breaks - 1) < 1e-9)) x_breaks <- sort(unique(c(x_breaks, 1)))

# -------------------------
# Left panel: forest plot
# -------------------------
p_forest <- ggplot2::ggplot(plot_df, ggplot2::aes(y = label)) +
  ggplot2::geom_vline(xintercept = 1, linetype = "dashed", linewidth = 0.5, colour = "grey35") +
  ggplot2::geom_errorbarh(ggplot2::aes(xmin = lcl, xmax = ucl), height = 0.18, linewidth = 0.8, colour = "black") +
  ggplot2::geom_point(ggplot2::aes(x = or), size = 2.6, colour = "black") +
  ggplot2::scale_x_log10(
    limits = c(x_left, x_right),
    breaks = x_breaks,
    labels = function(x) as.character(x)
  ) +
  ggplot2::scale_y_discrete(drop = FALSE, limits = levels(plot_df$label)) +
  ggplot2::labs(x = "Adjusted odds ratio (log scale)", y = NULL) +
  ggplot2::theme_classic(base_size = 11) +
  ggplot2::theme(
    axis.title.x = ggplot2::element_text(face = "bold"),
    axis.text.y  = ggplot2::element_text(size = 10),
    plot.background  = ggplot2::element_rect(fill = "white", colour = NA),
    panel.background = ggplot2::element_rect(fill = "white", colour = NA),
    plot.margin = ggplot2::margin(8, 6, 8, 8)
  )

# -------------------------
# Right panel: aligned text table
# FIX: use SAME plot_df and SAME y scale limits
# -------------------------
y_header <- length(levels(plot_df$label)) + 0.7

p_table <- ggplot2::ggplot(plot_df, ggplot2::aes(y = label)) +
  ggplot2::geom_text(ggplot2::aes(x = 0.0, label = orci_text), hjust = 0, size = 3.1) +
  ggplot2::geom_text(ggplot2::aes(x = 1.85, label = p_text), hjust = 0, size = 3.1) +
  ggplot2::annotate("text", x = 0.0, y = y_header, label = "Adjusted OR (95% CI)",
                    hjust = 0, fontface = "bold", size = 3.2) +
  ggplot2::annotate("text", x = 1.85, y = y_header, label = "P",
                    hjust = 0, fontface = "bold", size = 3.2) +
  ggplot2::scale_y_discrete(drop = FALSE, limits = levels(plot_df$label)) +
  ggplot2::coord_cartesian(xlim = c(0, 2.35), clip = "off") +
  ggplot2::theme_void(base_size = 11) +
  ggplot2::theme(
    plot.background  = ggplot2::element_rect(fill = "white", colour = NA),
    panel.background = ggplot2::element_rect(fill = "white", colour = NA),
    plot.margin = ggplot2::margin(8, 8, 8, 0)
  )

# -------------------------
# Combine into Figure 2
# -------------------------
fig2 <- (p_forest | p_table) +
  patchwork::plot_layout(widths = c(0.64, 0.36)) +
  patchwork::plot_annotation(
    title = "Optional Figure. Predictors of podium-to-print publication (adjusted model)",
    subtitle = sprintf("Multivariable logistic regression (complete-case N=%d; published n=%d)", N_model, events),
    caption = "Points denote adjusted odds ratios and horizontal lines denote 95% Wald confidence intervals."
  ) &
  ggplot2::theme(
    plot.title = ggplot2::element_text(face = "bold", size = 18, hjust = 0),
    plot.subtitle = ggplot2::element_text(size = 13, hjust = 0),
    plot.caption = ggplot2::element_text(size = 11, hjust = 0)
  )

# -------------------------
# Export
# -------------------------
ggplot2::ggsave(paste0(OUT_PREFIX, ".png"), fig2, width = 12.6, height = 6.6, dpi = 600, bg = "white")
ggplot2::ggsave(paste0(OUT_PREFIX, ".pdf"), fig2, width = 12.6, height = 6.6, bg = "white")

# Optional: write the plotted table
readr::write_csv(plot_df, paste0(OUT_PREFIX, "_table.csv"))

cat("Wrote:\n",
    "- ", OUT_PREFIX, ".png\n",
    "- ", OUT_PREFIX, ".pdf\n",
    "- ", OUT_PREFIX, "_table.csv\n", sep = "")

Wrote:
- /home/asegura/repos/aans-abstract-publication-2010-2019/outputs/supplemental_figure_adjusted_OR_forestplot_clean_aligned.png
- /home/asegura/repos/aans-abstract-publication-2010-2019/outputs/supplemental_figure_adjusted_OR_forestplot_clean_aligned.pdf
- /home/asegura/repos/aans-abstract-publication-2010-2019/outputs/supplemental_figure_adjusted_OR_forestplot_clean_aligned_table.csv


## Reproducibility
Record R session information used to generate outputs.

In [34]:
writeLines(capture.output(sessionInfo()), file.path(AANS_OUT_DIR, "sessionInfo.txt"))
cat("Wrote sessionInfo.txt\n")


Wrote sessionInfo.txt
